# VisionGuard — Tasks 1 and 2
This notebook contains the project code; no GitHub upload is needed to start.

**Storage:** Drive mounting is optional. If it fails, the notebook falls back to local runtime storage. Set `USE_GOOGLE_DRIVE = False` to skip the mount attempt. Download outputs before the runtime ends; local files are temporary.

**Before running:** choose **Runtime → Change runtime type → Python 3 → T4 GPU** (or another GPU).
Run cells in order. Do not use Run all until you have checked the folder labels in the dataset section.
Replace your roll number below. Use the same `<RollNumber>_A2_Ops` name on GitHub and DagsHub.

This notebook creates actual evidence when run. It contains no claimed dataset statistics or model scores in advance.
Full eight-trial tuning is required; do not report a shortened smoke test as final results.
The optional Grad-CAM bonus is outside this notebook's required scope.

In [ ]:
ROLL_NUMBER = "23L-0727"
assert ROLL_NUMBER != "REPLACE_WITH_YOUR_ROLL_NUMBER", "Enter your roll number first"
PROJECT_NAME = f"{ROLL_NUMBER}_A2_Ops"
SEED = 42
from pathlib import Path
import os
USE_GOOGLE_DRIVE = False  # Drive was full during the submitted run; download local evidence.
DRIVE_MOUNTED = False
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', timeout_ms=180000)
        DRIVE_MOUNTED = os.path.ismount('/content/drive') and Path('/content/drive/MyDrive').is_dir()
        if not DRIVE_MOUNTED:
            raise RuntimeError('Drive mount was not verified')
    except (ValueError, RuntimeError, OSError, ImportError) as exc:
        print(f'Google Drive unavailable: {exc}')
        print('Continuing with LOCAL runtime storage.')
WORK_DIR = Path('/content') / PROJECT_NAME
PROJECT_DIR = ((Path('/content/drive/MyDrive') / PROJECT_NAME) if DRIVE_MOUNTED
               else (Path('/content/visionguard_results') / PROJECT_NAME))
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
print('Project:', PROJECT_NAME)
print('Evidence folder:', PROJECT_DIR)
if not DRIVE_MOUNTED:
    print('WARNING: Local files are temporary. Download evidence before the runtime is deleted.')

## 1.1 — Isolated environment and package/GPU evidence
A fresh Colab runtime isolates the work from your laptop. The installation below keeps Colab's existing TensorFlow/CUDA stack, installs the other libraries, and records **exact installed versions**. For a separate local virtual environment, see `docs/LOCAL_SETUP.md` in the downloaded source bundle.
If Colab asks for a runtime restart, restart and rerun the setup cells before importing TensorFlow. GPU access depends on your Colab quota.

In [ ]:
%pip -q install "tensorboard>=2.16,<2.22" "keras-tuner>=1.4.7,<1.5" "kagglehub>=0.3,<1" "Pillow>=10,<13" "pandas>=2.2,<3.1" "matplotlib>=3.8,<4" "scikit-learn>=1.4,<2" "dvc[s3]>=3.50,<4"
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'Select a GPU runtime before the full training run.' 

## Write the bundled source files
Rerunning this cell restores the source; it does not delete datasets, checkpoints or results.

In [ ]:
BUNDLED_FILES = {'visionguard/__init__.py': '"""VisionGuard: assignment Tasks 1 and 2."""\n', 'visionguard/data.py': '"""Audit original files, create RGB training copies, and write disjoint split lists."""\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageOps\nfrom sklearn.model_selection import train_test_split\n\nLABELS = {\'real\': 0, \'ai\': 1}\nALIASES = {\'aiartdata\': \'ai\', \'realart\': \'real\', \'real\': \'real\', \'realimages\': \'real\', \'realimage\': \'real\',\n           \'human\': \'real\', \'art\': \'ai\', \'ai\': \'ai\', \'fake\': \'ai\',\n           \'aiimages\': \'ai\', \'aigenerated\': \'ai\', \'aigeneratedimages\': \'ai\',\n           \'fakeimages\': \'ai\', \'synthetic\': \'ai\'}\nEXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\', \'.webp\', \'.bmp\', \'.gif\', \'.tif\', \'.tiff\', \'.avif\'}\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef infer_class(path, mapping):\n    # The nearest recognized ancestor is the class; split/outer folders are ignored.\n    for part in reversed(path.parts[:-1]):\n        key = \'\'.join(c for c in part.lower() if c.isalnum())\n        if key in mapping:\n            return mapping[key]\n    raise ValueError(f\'No class folder recognized for {path}. Supply --class-map JSON.\')\n\n\ndef rgb_image(path):\n    with Image.open(path) as im:\n        im.seek(0)  # Animated files use their first frame only.\n        return ImageOps.exif_transpose(im).convert(\'RGB\')\n\n\ndef audit(root, out, mapping=None):\n    root, out = Path(root), Path(out)\n    if not root.is_dir():\n        raise FileNotFoundError(f\'Dataset directory does not exist: {root}\')\n    # Validate recognized image paths before hashing/decoding the full dataset.\n    effective_mapping = mapping or ALIASES\n    for candidate in root.rglob(\'*\'):\n        if (candidate.is_file() and candidate.suffix.lower() in EXTENSIONS\n                and not any(x.startswith(\'.\') for x in candidate.relative_to(root).parts)):\n            infer_class(candidate.relative_to(root), effective_mapping)\n    out.mkdir(parents=True, exist_ok=True)\n    mapping = mapping or ALIASES\n    rows, skipped = [], []\n    for p in sorted(root.rglob(\'*\')):\n        if not p.is_file() or any(x.startswith(\'.\') for x in p.relative_to(root).parts):\n            continue\n        rel = p.relative_to(root)\n        if p.suffix.lower() not in EXTENSIONS:\n            # Try opening unknown extensions too; record sidecars separately.\n            try:\n                with Image.open(p):\n                    pass\n            except Exception:\n                skipped.append(str(rel))\n                continue\n        cls = infer_class(rel, mapping)\n        row = dict(path=str(rel), class_name=cls, label=LABELS[cls],\n                   size_bytes=p.stat().st_size, sha256=file_hash(p), width=None,\n                   height=None, mode=None, format=None, pixel_sha256=None, error=\'\')\n        try:\n            with Image.open(p) as im:\n                row.update(width=im.width, height=im.height, mode=im.mode, format=im.format)\n                im.verify()\n            im = rgb_image(p)\n            im.load()\n            row[\'pixel_sha256\'] = hashlib.sha256(\n                f\'{im.width}x{im.height}:RGB:\'.encode() + im.tobytes()).hexdigest()\n        except Exception as e:\n            row[\'error\'] = f\'{type(e).__name__}: {e}\'\n        rows.append(row)\n    if not rows:\n        raise ValueError(\'No labeled images found. Check DATA_ROOT and class folder mapping.\')\n    df = pd.DataFrame(rows)\n    df.to_csv(out / \'audit.csv\', index=False)\n    valid = df[df.error.eq(\'\')].copy()\n    df[df.error.ne(\'\')].to_csv(out / \'corrupt.csv\', index=False)\n    df[df.duplicated(\'sha256\', keep=False)].to_csv(out / \'exact_duplicates.csv\', index=False)\n    valid[valid.duplicated(\'pixel_sha256\', keep=False)].to_csv(out / \'pixel_duplicates.csv\', index=False)\n    stats = valid.groupby(\'class_name\')[[\'width\', \'height\', \'size_bytes\']].agg([\'min\', \'median\', \'max\'])\n    stats.to_csv(out / \'summary_statistics.csv\')\n    counts = df.groupby(\'class_name\').agg(files=(\'path\', \'size\'), readable=(\'error\', lambda s: s.eq(\'\').sum()))\n    counts.to_csv(out / \'class_counts.csv\')\n    info = dict(total_files=len(df), readable_files=len(valid), corrupt_files=int(df.error.ne(\'\').sum()),\n                exact_duplicate_extra_files=int(df.duplicated(\'sha256\').sum()),\n                exact_duplicate_groups=int(df.loc[df.duplicated(\'sha256\', False), \'sha256\'].nunique()),\n                decoded_pixel_duplicate_extra_files=int(valid.duplicated(\'pixel_sha256\').sum()),\n                skipped_nonimage_files=skipped, label_mapping=LABELS)\n    (out / \'audit_summary.json\').write_text(json.dumps(info, indent=2))\n    print(counts.to_string())\n    print(stats.to_string())\n    print(json.dumps(info, indent=2))\n    return df\n\n\ndef eda(df, root, out, seed=42):\n    out = Path(out)\n    valid = df[df.error.eq(\'\')]\n    fig, axes = plt.subplots(2, 5, figsize=(14, 6), squeeze=False)\n    for row, cls in enumerate(LABELS):\n        sample = valid[valid.class_name.eq(cls)].sample(min(5, sum(valid.class_name.eq(cls))), random_state=seed)\n        for ax in axes[row]:\n            ax.axis(\'off\')\n        for ax, rec in zip(axes[row], sample.itertuples()):\n            ax.imshow(rgb_image(Path(root) / rec.path))\n            ax.set_title(f\'{cls}: {rec.width} x {rec.height}\', fontsize=9)\n    fig.suptitle(\'Original images: deterministic sample from each class\')\n    fig.tight_layout()\n    fig.savefig(out / \'samples.png\', dpi=160)\n    plt.close(fig)\n    for col in [\'width\', \'height\', \'size_bytes\']:\n        fig, ax = plt.subplots(figsize=(8, 4))\n        values = valid[col].to_numpy(float)\n        # Log-spaced bins retain visibility if there is a long tail.\n        lo, hi = max(values.min(), 1), max(values.max(), values.min() + 1)\n        bins = np.geomspace(lo, hi, 30)\n        for cls in LABELS:\n            ax.hist(valid.loc[valid.class_name.eq(cls), col], bins=bins, alpha=.5, label=cls)\n        ax.set_xscale(\'log\')\n        ax.set(xlabel=col + \' (log scale)\', ylabel=\'Image count\', title=f\'{col} by class\')\n        ax.legend()\n        fig.tight_layout()\n        fig.savefig(out / f\'{col}_histogram.png\', dpi=160)\n        plt.close(fig)\n\n\ndef prepare(df, root, processed, out, seed=42):\n    """Exclude corrupt/conflicting images and deduplicate BEFORE stratified splitting."""\n    out, processed = Path(out), Path(processed)\n    valid = df[df.error.eq(\'\')].copy()\n    conflicts = valid.groupby(\'pixel_sha256\').label.nunique()\n    bad_hashes = set(conflicts[conflicts.gt(1)].index)\n    conflict_rows = valid[valid.pixel_sha256.isin(bad_hashes)]\n    conflict_rows.to_csv(out / \'label_conflicts.csv\', index=False)\n    clean = valid[~valid.pixel_sha256.isin(bad_hashes)].drop_duplicates(\'pixel_sha256\').copy()\n    if set(clean.label) != {0, 1} or clean.groupby(\'label\').size().min() < 10:\n        raise ValueError(\'Need at least 10 distinct readable images per class for 70/15/15 splitting.\')\n    train, rest = train_test_split(clean, test_size=.30, random_state=seed, stratify=clean.label)\n    val, test = train_test_split(rest, test_size=.50, random_state=seed, stratify=rest.label)\n    processed.mkdir(parents=True, exist_ok=True)\n    records = []\n    for split, frame in [(\'train\', train), (\'val\', val), (\'test\', test)]:\n        frame = frame.sort_values(\'path\').copy()\n        frame[\'source_path\'] = frame[\'path\']\n        names = []\n        for rec in frame.itertuples():\n            dest = processed / rec.class_name / f\'{rec.pixel_sha256}.png\'\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            if not dest.exists():\n                im = rgb_image(Path(root) / rec.path)\n                # Fresh pixels discard inherited ICC/EXIF metadata. Atomic replacement\n                # prevents an interrupted conversion from leaving a partial final PNG.\n                clean_im = Image.frombytes(\'RGB\', im.size, im.tobytes())\n                temporary = dest.with_name(dest.name + \'.tmp\')\n                try:\n                    clean_im.save(temporary, format=\'PNG\')\n                    os.replace(temporary, dest)\n                finally:\n                    temporary.unlink(missing_ok=True)\n            names.append(str(dest.relative_to(processed)))\n        frame[\'path\'] = names\n        frame.to_csv(out / f\'{split}.csv\', index=False)\n        records.append(dict(split=split, files=len(frame), real=int(sum(frame.label == 0)), ai=int(sum(frame.label == 1))))\n    summary = dict(seed=seed, proportions=[.7, .15, .15], splits=records,\n                   excluded_label_conflicts=len(conflict_rows), retained_images=len(clean),\n                   duplicates_removed=len(valid) - len(conflict_rows) - len(clean),\n                   original_split_policy=\'Repartition globally after deduplication; do not mix with published-split results.\',\n                   limits=\'Exact decoded-pixel deduplication only; near duplicates and shared sources may remain.\')\n    (out / \'split_summary.json\').write_text(json.dumps(summary, indent=2))\n    (out / \'class_mapping.json\').write_text(json.dumps(LABELS, indent=2))\n    print(json.dumps(summary, indent=2))\n    return summary\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--root\', required=True)\n    p.add_argument(\'--out\', default=\'outputs/data\')\n    p.add_argument(\'--processed\', default=\'data/processed\')\n    p.add_argument(\'--class-map\', help=\'JSON mapping actual folder names to real or ai\')\n    p.add_argument(\'--seed\', type=int, default=42)\n    a = p.parse_args()\n    mapping = None\n    if a.class_map:\n        raw = json.loads(a.class_map)\n        if not set(raw.values()) <= set(LABELS):\n            raise ValueError(\'Mapping values must be real or ai\')\n        mapping = {\'\'.join(c for c in k.lower() if c.isalnum()): v for k, v in raw.items()}\n    df = audit(a.root, a.out, mapping)\n    eda(df, a.root, a.out, a.seed)\n    prepare(df, a.root, a.processed, a.out, a.seed)\n\nif __name__ == \'__main__\':\n    main()\n', 'visionguard/evaluation.py': '"""Validation-only threshold choice, a single held-out test pass, and error evidence."""\nimport json\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,\n                             roc_auc_score, confusion_matrix, roc_curve)\nfrom .data import rgb_image\n\n\ndef choose_threshold(y, probabilities, fp_cost=2., fn_cost=1.):\n    """Minimize 2*false accusations + missed AI on validation; AI is positive."""\n    y, probabilities = np.asarray(y).ravel(), np.asarray(probabilities).ravel()\n    if set(np.unique(y)) != {0, 1}:\n        raise ValueError(\'Both classes are required to select a threshold\')\n    # Every attainable classification boundary, including all-real and all-AI.\n    fpr, tpr, thresholds = roc_curve(y, probabilities, drop_intermediate=False)\n    fp = np.rint(fpr * sum(y == 0)).astype(int)\n    fn = np.rint((1-tpr) * sum(y == 1)).astype(int)\n    costs = fp_cost*fp + fn_cost*fn\n    tied = np.flatnonzero(costs == costs.min())\n    # Prefer the conservative higher threshold for an exact cost tie.\n    i = tied[0]\n    t = float(thresholds[i])\n    if not np.isfinite(t):\n        t = float(np.nextafter(probabilities.max(), np.inf))\n    table = pd.DataFrame(dict(threshold=thresholds, false_positive=fp, false_negative=fn, cost=costs))\n    return t, table\n\n\ndef metrics(y, p, threshold):\n    pred = np.asarray(p) >= threshold\n    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()\n    return dict(threshold=threshold, accuracy=float(accuracy_score(y, pred)),\n                precision=float(precision_score(y, pred, zero_division=0)),\n                recall=float(recall_score(y, pred, zero_division=0)),\n                f1=float(f1_score(y, pred, zero_division=0)), roc_auc=float(roc_auc_score(y, p)),\n                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))\n\n\ndef error_evidence(frame, p, threshold, root, out):\n    out = Path(out)\n    wrong = frame.copy()\n    wrong[\'p_ai\'] = p\n    wrong[\'predicted_label\'] = (p >= threshold).astype(int)\n    wrong = wrong[wrong.label.ne(wrong.predicted_label)].head(10).copy()\n    # Hypotheses must be written after inspecting the actual errors, not invented.\n    wrong[\'visual_observation\'] = \'\'\n    wrong[\'hypothesis_to_verify\'] = \'\'\n    wrong.to_csv(out / \'error_analysis.csv\', index=False)\n    fig, axes = plt.subplots(2, 5, figsize=(15, 7))\n    for ax in axes.flat:\n        ax.axis(\'off\')\n    for ax, rec in zip(axes.flat, wrong.itertuples()):\n        ax.imshow(rgb_image(Path(root) / rec.path))\n        ax.set_title(f\'True: {"AI" if rec.label else "Real"}\\nPred: {"AI" if rec.predicted_label else "Real"}; P(AI)={rec.p_ai:.3f}\', fontsize=9)\n    fig.suptitle(f\'Test errors: {len(wrong)} shown (up to 10 available)\')\n    fig.tight_layout()\n    fig.savefig(out / \'misclassified.png\', dpi=160)\n    plt.close(fig)\n    return len(wrong)\n\n\ndef evaluate_predictions(frame, p, threshold, root, out):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    y = frame.label.to_numpy()\n    result = metrics(y, p, threshold)\n    (out / \'metrics.json\').write_text(json.dumps(result, indent=2))\n    predictions = frame[[\'path\', \'source_path\', \'label\', \'sha256\', \'pixel_sha256\']].copy()\n    predictions[\'p_ai\'] = p\n    predictions[\'predicted_label\'] = (p >= threshold).astype(int)\n    predictions.to_csv(out / \'predictions.csv\', index=False)\n    fig, ax = plt.subplots(figsize=(5, 4))\n    cm = confusion_matrix(y, p >= threshold, labels=[0, 1])\n    ax.imshow(cm, cmap=\'Blues\')\n    for (i, j), value in np.ndenumerate(cm):\n        ax.text(j, i, str(value), ha=\'center\', va=\'center\')\n    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=[\'Real\', \'AI\'], yticklabels=[\'Real\', \'AI\'],\n           xlabel=\'Predicted\', ylabel=\'True\', title=\'Confusion matrix (AI is positive)\')\n    fig.tight_layout()\n    fig.savefig(out / \'confusion_matrix.png\', dpi=160)\n    plt.close(fig)\n    fpr, tpr, _ = roc_curve(y, p)\n    fig, ax = plt.subplots(figsize=(5, 4))\n    ax.plot(fpr, tpr, label=f\'AUC={result["roc_auc"]:.4f}\')\n    ax.plot([0, 1], [0, 1], \'--\', color=\'gray\')\n    ax.set(xlabel=\'False positive rate\', ylabel=\'True positive rate\', title=\'Test ROC curve\')\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(out / \'roc.png\', dpi=160)\n    plt.close(fig)\n    result[\'errors_shown\'] = error_evidence(frame, p, threshold, root, out)\n    return result\n', 'visionguard/training.py': '"""tf.data, model definitions, reproducible fitting, and saved training evidence."""\nimport json\nimport os\nimport random\nimport subprocess\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport tensorflow as tf\nfrom tensorflow import keras\n\n\ndef seed_everything(seed=42):\n    random.seed(seed)\n    np.random.seed(seed)\n    keras.utils.set_random_seed(seed)\n    tf.config.experimental.enable_op_determinism()\n\n\ndef environment(out):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    record = dict(timestamp_utc=datetime.now(timezone.utc).isoformat(), python=sys.version,\n                  tensorflow=tf.__version__, keras=keras.__version__,\n                  gpu=[x.name for x in tf.config.list_physical_devices(\'GPU\')],\n                  build=tf.sysconfig.get_build_info(), platform=sys.platform)\n    (out / \'environment.json\').write_text(json.dumps(record, indent=2, default=str))\n    (out / \'requirements-lock.txt\').write_text(subprocess.check_output(\n        [sys.executable, \'-m\', \'pip\', \'freeze\'], text=True))\n    print(json.dumps(record, indent=2, default=str))\n    return record\n\n\ndef dataset(split_csv, root, size=224, batch=32, training=False, seed=42):\n    frame = pd.read_csv(split_csv)\n    paths = [str(Path(root) / p) for p in frame.path]\n    ds = tf.data.Dataset.from_tensor_slices((paths, frame.label.to_numpy(np.float32)))\n    def decode(path, label):\n        im = tf.io.decode_png(tf.io.read_file(path), channels=3)\n        # Letterboxing preserves aspect ratio. Padding is zero; no geometric distortion.\n        im = tf.image.resize_with_pad(im, size, size, method=\'bilinear\', antialias=True)\n        return im, tf.reshape(label, [1])\n    opts = tf.data.Options()\n    opts.experimental_deterministic = True\n    # Cache deterministic resized pixels, then reshuffle each training epoch.\n    # The assignment\'s 680 training images occupy about 410 MB at 224px.\n    # Augmentation stays inside the model and is never cached.\n    ds = ds.map(decode, num_parallel_calls=2).with_options(opts).cache()\n    if training:\n        ds = ds.shuffle(len(frame), seed=seed, reshuffle_each_iteration=True)\n    return ds.batch(batch).prefetch(1)\n\n\ndef augmentation(seed=42):\n    return keras.Sequential([\n        keras.layers.RandomFlip(\'horizontal\', seed=seed),\n        keras.layers.RandomRotation(.025, fill_mode=\'reflect\', seed=seed+1),\n        keras.layers.RandomZoom(.08, fill_mode=\'reflect\', seed=seed+2),\n        keras.layers.RandomContrast(.08, seed=seed+3),\n    ], name=\'train_only_augmentation\')\n\n\ndef compile_model(model, lr):\n    model.compile(optimizer=keras.optimizers.Adam(lr), loss=\'binary_crossentropy\',\n                  metrics=[keras.metrics.BinaryAccuracy(name=\'accuracy\'),\n                           keras.metrics.AUC(name=\'auc\'),\n                           keras.metrics.Precision(name=\'precision\'), keras.metrics.Recall(name=\'recall\')])\n    return model\n\n\ndef baseline(size=224, seed=42):\n    inputs = keras.Input((size, size, 3), name=\'rgb_pixels_0_255\')\n    x = augmentation(seed)(inputs)\n    x = keras.layers.Rescaling(1/255., name=\'scale_0_1\')(x)\n    for i, filters in enumerate([32, 64, 128], 1):\n        x = keras.layers.Conv2D(filters, 3, padding=\'same\', use_bias=False, name=f\'conv_{i}\')(x)\n        x = keras.layers.BatchNormalization(name=f\'bn_{i}\')(x)\n        x = keras.layers.Activation(\'relu\', name=f\'relu_{i}\')(x)\n        x = keras.layers.MaxPooling2D(name=f\'pool_{i}\')(x)\n    x = keras.layers.GlobalAveragePooling2D()(x)\n    x = keras.layers.Dense(64, activation=\'relu\')(x)\n    x = keras.layers.Dropout(.4)(x)\n    output = keras.layers.Dense(1, activation=\'sigmoid\', name=\'p_ai\')(x)\n    return compile_model(keras.Model(inputs, output, name=\'scratch_cnn\'), 1e-3)\n\n\ndef transfer(size=224, dropout=.3, seed=42, weights=\'imagenet\'):\n    inputs = keras.Input((size, size, 3), name=\'rgb_pixels_0_255\')\n    x = augmentation(seed)(inputs)\n    # Exactly equivalent to MobileNetV2 preprocess_input: (pixels / 127.5) - 1.\n    x = keras.layers.Rescaling(1/127.5, offset=-1, name=\'mobilenet_preprocess\')(x)\n    backbone = keras.applications.MobileNetV2(include_top=False, weights=weights, input_shape=(size, size, 3))\n    backbone.trainable = False\n    # Keep BN moving statistics fixed during both stages.\n    x = backbone(x, training=False)\n    x = keras.layers.GlobalAveragePooling2D()(x)\n    x = keras.layers.Dropout(dropout, name=\'head_dropout\')(x)\n    output = keras.layers.Dense(1, activation=\'sigmoid\', name=\'p_ai\')(x)\n    return compile_model(keras.Model(inputs, output, name=\'mobilenetv2_classifier\'), 1e-3)\n\n\ndef unfreeze_last_block(model, count=None):\n    backbone = model.get_layer(\'mobilenetv2_1.00_224\') if any(\n        l.name == \'mobilenetv2_1.00_224\' for l in model.layers) else next(\n        l for l in model.layers if isinstance(l, keras.Model) and \'mobilenetv2\' in l.name)\n    backbone.trainable = True\n    eligible = [l for l in backbone.layers if\n                (l.name.startswith(\'block_16_\') or l.name == \'Conv_1\')\n                and l.weights and not isinstance(l, keras.layers.BatchNormalization)]\n    chosen = eligible if count is None else eligible[-count:]\n    names = {l.name for l in chosen}\n    for layer in backbone.layers:\n        layer.trainable = layer.name in names\n    return [l.name for l in chosen]\n\n\ndef layer_evidence(model, out):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    lines = []\n    model.summary(print_fn=lambda s, **kw: lines.append(s), expand_nested=True)\n    (out / \'model_summary.txt\').write_text(\'\\n\'.join(lines))\n    rows, trainable = [], []\n    def walk(layers, prefix=\'\'):\n        for l in layers:\n            name = prefix + l.name\n            rows.append(dict(layer=name, type=type(l).__name__,\n                             output_shape=str(getattr(l, \'output\', \'\') .shape if hasattr(getattr(l, \'output\', None), \'shape\') else \'\'),\n                             parameters=l.count_params(), trainable=l.trainable))\n            if isinstance(l, keras.Model):\n                walk(l.layers, name + \'/\')\n            elif l.trainable and l.trainable_weights:\n                trainable.append(name)\n    walk(model.layers)\n    pd.DataFrame(rows).to_csv(out / \'layers.csv\', index=False)\n    (out / \'trainable_layers.json\').write_text(json.dumps(trainable, indent=2))\n    print(\'\\n\'.join(lines))\n\n\ndef fit_stage(model, train, val, out, epochs=15):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    layer_evidence(model, out)\n    callbacks = [keras.callbacks.EarlyStopping(monitor=\'val_loss\', patience=4, restore_best_weights=True),\n                 keras.callbacks.ModelCheckpoint(str(out / \'best.keras\'), monitor=\'val_loss\', save_best_only=True),\n                 keras.callbacks.ReduceLROnPlateau(monitor=\'val_loss\', factor=.5, patience=2, min_lr=1e-7),\n                 keras.callbacks.CSVLogger(str(out / \'epochs.csv\'))]\n    h = model.fit(train, validation_data=val, epochs=epochs, callbacks=callbacks, verbose=2).history\n    best = int(np.argmin(h[\'val_loss\'])) + 1\n    (out / \'history.json\').write_text(json.dumps(h, indent=2))\n    info = dict(best_epoch=best, best_val_loss=float(min(h[\'val_loss\'])), epochs_run=len(h[\'loss\']))\n    (out / \'best_epoch.json\').write_text(json.dumps(info, indent=2))\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n    for ax, metric in zip(axes, [\'loss\', \'accuracy\']):\n        ax.plot(range(1, len(h[metric])+1), h[metric], marker=\'o\', label=\'train\')\n        ax.plot(range(1, len(h[metric])+1), h[\'val_\'+metric], marker=\'o\', label=\'validation\')\n        ax.axvline(best, linestyle=\'--\', color=\'gray\', label=f\'Best val-loss epoch: {best}\')\n        ax.set(xlabel=\'Epoch\', ylabel=metric, title=out.name)\n        ax.legend()\n    fig.tight_layout()\n    fig.savefig(out / \'curves.png\', dpi=160)\n    plt.close(fig)\n    return keras.models.load_model(out / \'best.keras\'), info\n\n\ndef augmented_batch(ds, out, seed=42):\n    images, labels = next(iter(ds))\n    aug = augmentation(seed)(images, training=True).numpy()\n    n = min(12, len(aug))\n    fig, axes = plt.subplots(3, 4, figsize=(10, 8))\n    for i, ax in enumerate(axes.flat):\n        ax.axis(\'off\')\n        if i < n:\n            ax.imshow(np.clip(aug[i], 0, 255).astype(\'uint8\'))\n            ax.set_title(\'AI\' if labels[i].numpy().item() else \'Real\')\n    fig.suptitle(\'Training augmentation preview (not used for validation/test)\')\n    fig.tight_layout()\n    fig.savefig(out, dpi=160)\n    plt.close(fig)\n', 'visionguard/tuning.py': '"""Task 2.7: Fashion-MNIST toy search and >=8 VisionGuard trials."""\nimport json\nfrom pathlib import Path\nimport keras_tuner as kt\nimport numpy as np\nimport pandas as pd\nfrom tensorflow import keras\nfrom .training import transfer, compile_model, unfreeze_last_block, seed_everything\n\n\ndef export_trials(tuner, out):\n    rows = []\n    for trial in tuner.oracle.trials.values():\n        rows.append(dict(trial_id=trial.trial_id, status=trial.status,\n                         val_loss=trial.score, **trial.hyperparameters.values))\n    df = pd.DataFrame(rows).sort_values(\'val_loss\')\n    df.to_csv(Path(out) / \'trials.csv\', index=False)\n    winner = tuner.oracle.get_best_trials(1)[0]\n    (Path(out) / \'best_trial.json\').write_text(json.dumps(dict(\n        trial_id=winner.trial_id, best_epoch=int(winner.best_step)+1,\n        val_loss=winner.score), indent=2))\n    best = tuner.get_best_hyperparameters(1)[0].values\n    (Path(out) / \'best_hyperparameters.json\').write_text(json.dumps(best, indent=2))\n    print(df.to_string(index=False))\n    return df, best\n\n\ndef toy(out=\'outputs/tuner_toy\', epochs=5, trials=8, seed=42):\n    seed_everything(seed)\n    (x, y), _ = keras.datasets.fashion_mnist.load_data()\n    # Use a fixed stratified subset, leave the built-in test set unused.\n    from sklearn.model_selection import train_test_split\n    ix, iv = train_test_split(np.arange(len(y)), train_size=10000, test_size=2000,\n                             stratify=y, random_state=seed)\n    x = x.astype(\'float32\')/255.\n    def build(hp):\n        model = keras.Sequential([keras.Input((28, 28)), keras.layers.Flatten(),\n                 keras.layers.Dense(hp.Choice(\'units\', [32, 64, 128]), activation=\'relu\'),\n                 keras.layers.Dense(10, activation=\'softmax\')])\n        model.compile(keras.optimizers.Adam(hp.Choice(\'learning_rate\', [1e-2, 1e-3, 1e-4])),\n                      loss=\'sparse_categorical_crossentropy\', metrics=[\'accuracy\'])\n        return model\n    tuner = kt.RandomSearch(build, objective=\'val_loss\', max_trials=trials,\n                            directory=out, project_name=\'search\', seed=seed, overwrite=False)\n    tuner.search(x[ix], y[ix], validation_data=(x[iv], y[iv]), epochs=epochs,\n                 callbacks=[keras.callbacks.EarlyStopping(monitor=\'val_loss\', patience=2)], verbose=2)\n    return export_trials(tuner, out)\n\n\ndef visionguard(train, val, frozen_checkpoint, out=\'outputs/tuner\', size=224,\n                epochs=5, trials=8, seed=42):\n    if trials < 8:\n        raise ValueError(\'Assignment requires at least eight trials\')\n    seed_everything(seed)\n    def build(hp):\n        # Every trial starts from the SAME trained frozen-head checkpoint.\n        model = transfer(size, dropout=hp.Choice(\'dropout\', [.2, .4, .5]), seed=seed, weights=None)\n        model.load_weights(frozen_checkpoint)\n        unfreeze_last_block(model, hp.Choice(\'unfrozen_layers\', [1, 2, 4]))\n        return compile_model(model, hp.Choice(\'learning_rate\', [1e-5, 3e-5, 1e-4]))\n    tuner = kt.RandomSearch(build, objective=\'val_loss\', max_trials=trials,\n                            directory=out, project_name=\'search\', seed=seed, overwrite=False,\n                            max_consecutive_failed_trials=2)\n    tuner.search(train, validation_data=val, epochs=epochs, callbacks=[\n        keras.callbacks.EarlyStopping(monitor=\'val_loss\', patience=3),\n        keras.callbacks.ReduceLROnPlateau(monitor=\'val_loss\', factor=.5, patience=2)], verbose=2)\n    table, best = export_trials(tuner, out)\n    if sum(table.status.eq(\'COMPLETED\')) < trials:\n        raise RuntimeError(\'Fewer than eight completed trials; inspect failed trials before submission.\')\n    # Descriptive marginal associations, not causal importance in a small random search.\n    notes = {}\n    for col in [\'learning_rate\', \'dropout\', \'unfrozen_layers\']:\n        group = table[table.status.eq(\'COMPLETED\')].groupby(col).val_loss.agg([\'mean\', \'count\'])\n        group.to_csv(Path(out) / f\'{col}_association.csv\')\n        notes[col] = float(group[\'mean\'].max() - group[\'mean\'].min())\n    (Path(out) / \'descriptive_effect_ranges.json\').write_text(json.dumps(notes, indent=2))\n    # Retain the actual winning trial checkpoint; no test data used.\n    model = tuner.get_best_models(1)[0]\n    model.save(Path(out) / \'best.keras\')\n    return model, best\n', 'scripts/__init__.py': '"""Command-line entry points for the assignment workflows."""\n', 'scripts/build_report.py': '"""Build the Tasks 1-2 PDF from actual exported Colab evidence."""\nimport json\nimport csv\nimport subprocess\nfrom pathlib import Path\nfrom html import escape\nfrom reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak, KeepTogether\nfrom reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle\nfrom reportlab.lib import colors\nfrom reportlab.lib.enums import TA_LEFT\nfrom reportlab.lib.utils import ImageReader\n\nROOT = Path(__file__).resolve().parents[1]\nE = ROOT/\'outputs/23L-0727_A2_Ops\'\nC = E/\'colab\'\nT = C/\'training_cached\'\nD = E/\'data_evidence\'\nS = E/\'screenshots\'\nOUT = ROOT/\'submission/23L-0727_VisionGuard.pdf\'\nstyles = getSampleStyleSheet()\nstyles.add(ParagraphStyle(name=\'BodyVG\',fontName=\'Helvetica\',fontSize=9.5,leading=13,spaceAfter=7,textColor=colors.HexColor(\'#26354a\')))\nstyles.add(ParagraphStyle(name=\'CaptionVG\',fontSize=8,leading=10,spaceBefore=4,spaceAfter=10,textColor=colors.HexColor(\'#546378\')))\nstyles.add(ParagraphStyle(name=\'TitleVG\',fontName=\'Helvetica-Bold\',fontSize=30,leading=35,spaceAfter=18,textColor=colors.HexColor(\'#122d4d\')))\nstyles.add(ParagraphStyle(name=\'HVG\',fontName=\'Helvetica-Bold\',fontSize=17,leading=21,spaceBefore=5,spaceAfter=12,textColor=colors.HexColor(\'#122d4d\')))\nstyles.add(ParagraphStyle(name=\'SmallVG\',fontName=\'Helvetica\',fontSize=7.5,leading=10,spaceAfter=5))\nstory=[]\nfig=0\n\ndef js(p): return json.loads(Path(p).read_text())\ndef rows(p):\n    with open(p,newline=\'\') as f: return list(csv.DictReader(f))\ndef para(text,style=\'BodyVG\'): story.append(Paragraph(text,styles[style]))\ndef heading(text): para(text,\'HVG\')\ndef page(text):\n    if story: story.append(PageBreak())\n    heading(text)\ndef table(data,widths=None,small=False):\n    sty=styles[\'SmallVG\' if small else \'BodyVG\']\n    wrapped=[[Paragraph(escape(str(v)).replace(\'\\n\',\'<br/>\'),sty) for v in row] for row in data]\n    tb=Table(wrapped,colWidths=widths,repeatRows=1,hAlign=\'LEFT\')\n    tb.setStyle(TableStyle([(\'BACKGROUND\',(0,0),(-1,0),colors.HexColor(\'#e5edf6\')),(\'VALIGN\',(0,0),(-1,-1),\'TOP\'),(\'LINEBELOW\',(0,0),(-1,0),.7,colors.HexColor(\'#9aadc3\')),(\'LINEBELOW\',(0,1),(-1,-1),.25,colors.HexColor(\'#dbe2ea\')),(\'LEFTPADDING\',(0,0),(-1,-1),6),(\'RIGHTPADDING\',(0,0),(-1,-1),6),(\'TOPPADDING\',(0,0),(-1,-1),5),(\'BOTTOMPADDING\',(0,0),(-1,-1),5)]))\n    story.extend([tb,Spacer(1,8)])\ndef image(path,caption,maxh=270):\n    global fig\n    path=Path(path)\n    if not path.exists(): raise FileNotFoundError(path)\n    iw,ih=ImageReader(str(path)).getSize()\n    scale=min(493/iw,maxh/ih)\n    fig+=1\n    story.append(KeepTogether([Image(str(path),width=iw*scale,height=ih*scale,hAlign=\'CENTER\'),Paragraph(f\'Figure {fig}. \'+caption,styles[\'CaptionVG\'])]))\ndef shot(name,caption,maxh=360): image(S/name,caption,maxh)\ndef footer(canvas,doc):\n    canvas.saveState()\n    w,h=doc.pagesize\n    canvas.setStrokeColor(colors.HexColor(\'#b7c8dc\'));canvas.line(48,42,w-48,42)\n    canvas.setFont(\'Helvetica\',8);canvas.setFillColor(colors.HexColor(\'#546378\'))\n    canvas.drawString(48,29,\'23L-0727_A2_Ops  |  VisionGuard  |  Tasks 1 and 2\')\n    canvas.drawRightString(w-48,29,str(doc.page));canvas.restoreState()\n\ndef build():\n    assert (T/\'COMPLETE.json\').exists(), \'Full real-data run is required before report generation\'\n    selection=js(T/\'selection_before_test.json\')\n    metrics={m:js(T/f\'evaluation/{m}/metrics.json\') for m in [\'baseline\',\'transfer\']}\n    env=js(T/\'environment.json\')\n    links=js(E/\'submission_metadata.json\')\n    para(\'VISIONGUARD\',\'TitleVG\')\n    para(\'AI-generated vs real image classification\',\'HVG\')\n    para(\'Task 1: Data Understanding and Versioning<br/>Task 2: Model Development with TensorFlow/Keras\')\n    story.append(Spacer(1,24))\n    table([[\'Submission detail\',\'Value\'],[\'Name\',links.get(\'name\',\'Hayyan Haider\')],[\'Roll number\',\'23L-0727\'],[\'Project\',\'23L-0727_A2_Ops\'],[\'Date\',\'6 November 2026 (Asia/Karachi)\'],[\'Graded source commit\',links[\'commit\']],[\'GitHub\',links[\'github\']],[\'DagsHub\',links[\'dagshub\']],[\'Colab notebook\',\'https://colab.research.google.com/drive/1DUeUYKEAaNH7_sl6O8K48r6MESux_Htl\'],[\'Scope\',\'Tasks 1 and 2 only. Deployment/API work belongs to later tasks.\']],[116,377])\n    para(\'<b>AI assistance disclosure.</b> OpenAI Codex helped prepare and repair the audit and training code, operated the Colab experiment, inspected evidence, and drafted this report. The reported measurements come from the actual assignment dataset and saved experiment outputs. The student should review the code, results, and interpretations before submission.\')\n    para(\'The experiment uses folder labels (AI=1, real=0). These labels are dataset annotations; a model score is not proof of image authenticity. All test figures reuse one inference pass per final model.\')\n\n    page(\'Task 1 - Data Understanding and Versioning\')\n    heading(\'1.1 - Environment and GPU\')\n    para(\'The local project uses an isolated Python virtual environment, separating its dependencies from system Python. The graded model experiment ran in a separate Google Colab runtime with an NVIDIA T4 GPU. TensorFlow GPU detection and the runtime/package record are shown below; exact installed package versions are supplied in requirements-lock.txt.\')\n    table([[\'Setting\',\'Observed value\'],[\'Python\',env[\'python\'].split(\'\\n\')[0]],[\'TensorFlow / Keras\',env[\'tensorflow\']+\' / \'+env[\'keras\']],[\'GPU devices\',\', \'.join(env[\'gpu\'])],[\'CUDA / cuDNN\',str(env[\'build\'].get(\'cuda_version\'))+\' / \'+str(env[\'build\'].get(\'cudnn_version\'))],[\'Random seed / image size / batch\',\'42 / 224 x 224 RGB / 32\']],[135,358])\n    shot(\'environment.png\',\'Actual Colab package versions and GPU evidence, with project context.\',340)\n    para(\'<b>Commands and choices.</b> python -m venv creates isolation; pip installs the declared dependencies. TensorFlow list_physical_devices checks GPU visibility; pip freeze records exact versions. Keeping Colab\\\'s supplied TensorFlow/CUDA combination avoids replacing its working GPU stack.\')\n    para(\'<b>Observed issue.</b> Google Drive was full, preventing notebook saves. Training checkpoints and outputs were therefore written under /content/23L-0727_evidence and downloaded to the local submission bundle. Runtime-local files alone are not persistent backups.\')\n\n    page(\'1.2 - Original dataset audit\')\n    para(\'The script recursively walks the original Kaggle files. For each candidate image it records class, file size, width, height, colour mode, format, byte SHA-256, decoded RGB pixel SHA-256, and any decoding error. Pillow verify plus a full decode checks readability. The nearest recognized parent folder assigns the label: AiArtData is AI and RealArt is real.\')\n    table([[\'Class\',\'Files\',\'Readable\',\'Width min/median/max\',\'Height min/median/max\'],[\'AI\',\'538\',\'538\',\'225 / 1028.5 / 6000\',\'135 / 800 / 4001\'],[\'Real\',\'435\',\'435\',\'183 / 940 / 8495\',\'158 / 720 / 7091\']],[42,40,53,179,179],True)\n    table([[\'Class\',\'File bytes: minimum\',\'Median\',\'Maximum\'],[\'AI\',\'11,738\',\'199,802.5\',\'5,626,183\'],[\'Real\',\'11,311\',\'144,982\',\'10,195,352\']],[60,145,144,144])\n    para(\'There are 973 readable images and no unreadable candidates. One duplicate group contributes one extra file; decoded-pixel deduplication also removes one extra image. No cross-label pixel conflicts were found. AI images account for 55.3% of the original collection, so accuracy should be considered alongside precision, recall, F1, and ROC-AUC.\')\n    table([[\'Split\',\'Real\',\'AI\',\'Total\'],[\'Train\',\'304\',\'376\',\'680\'],[\'Validation\',\'65\',\'81\',\'146\'],[\'Test\',\'66\',\'80\',\'146\']],[123,123,123,124])\n    para(\'Deduplication happens before a seeded, stratified 70/15/15 split. Split lists retain original paths and hashes, making membership auditable. The retained pixel-hash sets are disjoint across train/validation/test. This only detects identical images: near duplicates and shared web sources may still leak visual information.\')\n    shot(\'audit.png\',\'Executed audit and split evidence for the assignment dataset.\',295)\n\n    page(\'1.3 - Visual EDA: sample grid and widths\')\n    image(D/\'samples.png\',\'Deterministic original-image samples from each class (seed 42).\',235)\n    para(links[\'sample_interpretation\'])\n    image(D/\'width_histogram.png\',\'Original image width distribution by class; horizontal axis is logarithmic.\',220)\n    para(\'Median width is 1028.5 pixels for AI images and 940 pixels for real images, with substantial overlap. Real images have the more extreme maximum (8495 versus 6000 pixels). Resolution is therefore a possible source shortcut and compute cost; resizing makes tensor shapes consistent but does not establish that the detector has learned generation artifacts.\')\n\n    page(\'1.3 - Visual EDA: heights and file sizes\')\n    image(D/\'height_histogram.png\',\'Original image heights by class on a logarithmic axis.\',230)\n    para(\'Median height is 800 pixels for AI images and 720 for real images. Both distributions span small and large images, and real images reach 7091 pixels compared with 4001 for AI. A fixed 224-pixel input reduces memory use, while potentially discarding fine forensic details.\')\n    image(D/\'size_bytes_histogram.png\',\'Original file-size distributions by class on a logarithmic axis.\',230)\n    para(\'The AI median file size is about 199.8 kB, compared with 145.0 kB for real images, although the largest real file is about 10.2 MB. These are original compressed file sizes, not processed PNG sizes or tensor memory. Compression format and source quality can correlate with class, so file size alone would be an unreliable authenticity signal.\')\n\n    page(\'1.4 - Git and DVC: toy round trip\')\n    para(\'A new empty directory was initialized with Git and DVC. tiny.csv initially contained two labeled rows; dvc add created its pointer, and Git committed the metadata. A third row was added and committed as a second version. Both data versions were pushed to a configured local-folder DVC remote, which the assignment permits.\')\n    toy=js(E/\'dvc_toy/result.json\')\n    table([[\'Toy evidence\',\'Observed value\'],[\'Version 1 commit\',toy[\'first_commit\']],[\'Version 2 commit\',toy[\'second_commit\']],[\'Restored version\',toy[\'restored_commit\']],[\'Recovery assertion\',str(toy[\'verified\'])]],[112,381],True)\n    para(\'git checkout selected the first commit and dvc checkout restored its two-row content. The toy CSV and its local DVC cache were then removed. dvc pull recovered the older bytes from the remote, and the script asserted exact content equality. Deleting the cache makes this stronger than recovering from a local cached copy.\')\n    shot(\'dvc_toy.png\',\'Both toy Git commits published on the DagsHub dvc-toy branch. Recovery is verified in the accompanying transcript and result.json.\',350)\n    para(\'git init starts metadata history; dvc init creates DVC configuration; dvc add hashes data and writes pointers; dvc remote add chooses object storage; dvc push uploads objects; dvc pull retrieves the objects named by the checked-out pointer. The two toy commits are also published to the DagsHub project as described in the accompanying repository evidence.\')\n\n    page(\'1.4 - Apply DVC to the real dataset\')\n    para(\'The original dataset, normalized RGB copies, and split manifests were tracked separately with DVC. dvc push uploaded their objects to the persistent local remote under versioned_storage/dvc_remote. Git stores the small pointers and configuration; a bare Git backup preserves metadata independently of the working tree.\')\n    pointer=(ROOT/\'data/raw.dvc\').read_text()\n    para(\'<b>Actual raw pointer:</b><br/>\'+escape(pointer).replace(\'\\n\',\'<br/>\'),\'SmallVG\')\n    para(\'The pointer contains the content checksum, byte size, file count, hash algorithm, and path. For a directory, its .dir object identifies the file manifest. The audit uses SHA-256 for source identity; DVC\\\'s content checksum serves its own storage protocol. Large images remain outside Git because binary versions inflate history and do not produce useful text diffs.\')\n    para(\'The raw DVC pointer counts 975 files, while the image audit counts 973 candidates: DVC also includes two hidden .DS_Store files ignored by the image audit. This difference does not indicate two unreadable images.\')\n    shot(\'dvc_dataset.png\',\'Browser view of the actual saved raw pointer and freshly executed DVC remote status. The log file is included in the evidence bundle.\',300)\n    shot(\'dagshub.png\',\'DagsHub repository and published version history.\',240)\n    para(\'The local remote satisfies the permitted storage option. Reproducing on another computer requires access to that remote or an exported copy and an updated remote path; a Git clone alone cannot reconstruct excluded image bytes.\')\n\n    page(\'Task 2 - Model Development with TensorFlow/Keras\')\n    heading(\'2.1 - Input pipeline and augmentation\')\n    para(\'Saved CSV split lists supply file paths and float labels. tf.data reads each RGB PNG, decodes three channels, resizes with bilinear interpolation and antialiasing, and pads to 224 x 224 without stretching. This matches MobileNetV2\\\'s standard input and limits GPU memory, at the cost of losing small artifacts. Padding can itself become a shortcut if aspect ratios correlate with labels.\')\n    para(\'The deterministic resized tensors are cached in memory before training shuffle. Training reshuffles each epoch with seed 42; batches contain up to 32 examples, and one batch is prefetched. Decode parallelism is capped at two to limit memory. Cache avoids repeatedly decoding large PNGs, and does not cache augmentation. Validation/test preserve manifest order and use no augmentation.\')\n    para(\'Training-only augmentation lives inside each model: horizontal flips, rotation up to 0.025 turns, zoom factor 0.08, and contrast factor 0.08. Reflect padding reduces blank fill during geometric augmentation. Keras disables these random transforms during evaluation; saved preprocessing therefore travels with the model.\')\n    image(T/\'augmented_batch.png\',\'A genuine augmented training batch. Labels come from the manifest.\',355)\n    para(\'<b>Observed repair.</b> One processed PNG failed TensorFlow decoding. Its source byte hash and decoded pixel hash were checked against the original audit, then its RGB pixels were re-encoded with TensorFlow. All 972 split images passed decoding and pixel-hash verification; no examples or labels were dropped.\')\n\n    page(\'2.2 - Scratch CNN architecture\')\n    para(\'The scratch network has three convolution blocks (32, 64, 128 filters). Each 3 x 3 convolution shares local feature weights; BatchNorm stabilizes activations, ReLU adds nonlinearity, and max pooling reduces spatial size and computation. Global average pooling avoids a very large flattened classifier. A 64-unit ReLU layer combines channels; dropout 0.4 regularizes, and one sigmoid produces an AI score.\')\n    layers=rows(T/\'baseline/layers.csv\')\n    table([[\'Layer\',\'Output shape\',\'Parameters\']]+[[r[\'layer\'],r[\'output_shape\'],f"{int(r[\'parameters\']):,}"] for r in layers if \'/\' not in r[\'layer\']],[214,179,100],True)\n    para(\'The loss is binary cross entropy because the target is binary and the output is a sigmoid. Adam starts at 0.001, providing adaptive updates. Training monitors binary accuracy, AUC, precision, and recall; the latter classification metrics use 0.5 during fit. Final metrics instead use thresholds chosen solely from validation. Sigmoid scores are not guaranteed calibrated authenticity probabilities.\')\n    shot(\'model_summary.png\',\'Executed scratch CNN model summary with output shapes and parameter counts.\',250)\n\n    page(\'2.3 - Transfer learning in two stages\')\n    para(\'MobileNetV2 uses ImageNet pretrained weights with include_top=False. The added classifier consists of global average pooling, dropout 0.3, and one sigmoid. Embedded rescaling maps 0-255 pixels to [-1,1], matching MobileNetV2 preprocessing. ImageNet features provide a useful initialization, while the authenticity task differs from the original object classification task.\')\n    para(\'Stage 1 freezes the backbone and trains the head with Adam at 0.001. Stage 2 unfreezes only the last block\\\'s weighted non-BatchNorm layers and the terminal convolution, then recompiles with learning rate 0.00001 (100 times smaller). BatchNorm remains frozen and the backbone is called in inference mode to preserve its moving statistics.\')\n    for stage in [\'head\',\'finetune\']:\n        image(T/stage/\'curves.png\',f\'{stage}: loss and accuracy curves, with minimum validation-loss epoch marked.\',190)\n        names=js(T/stage/\'trainable_layers.json\')\n        para(\'<b>\'+stage+\' trainable weighted layers:</b> \'+\', \'.join(names),\'SmallVG\')\n    para(\'The best validation loss fell from 0.53795 for the frozen head to 0.49179 after last-block adaptation. This supports useful adaptation on this validation split, while the later tuner winner achieved 0.47383. The frozen-head best epoch was the budget limit (10), so early stopping did not establish convergence for that stage.\')\n\n    page(\'2.4 - Training discipline and checkpoints\')\n    para(\'Python, NumPy, and TensorFlow seeds are fixed to 42, and TensorFlow deterministic operations are enabled. Exact reproducibility still depends on hardware, CUDA, and package versions. A checkpoint is saved only when validation loss improves. Early stopping uses patience 4 and restores best weights; ReduceLROnPlateau halves the learning rate after 2 stagnant epochs with minimum 1e-7.\')\n    table([[\'Stage\',\'Epoch budget\',\'Epochs run\',\'Best epoch\',\'Best validation loss\']]+[[stage,budget,js(T/stage/\'best_epoch.json\')[\'epochs_run\'],js(T/stage/\'best_epoch.json\')[\'best_epoch\'],f"{js(T/stage/\'best_epoch.json\')[\'best_val_loss\']:.5f}"] for stage,budget in [(\'baseline\',20),(\'head\',10),(\'finetune\',15)]],[90,85,85,85,148])\n    image(T/\'baseline/curves.png\',\'Scratch CNN training curves, with the selected validation-loss checkpoint marked.\',205)\n    para(\'Scratch training loss generally decreases, while validation loss improves only to epoch 6 and worsens later. Training accuracy rises above validation accuracy, which stays near the class-majority level. This gap is consistent with overfitting or dataset mismatch; the curves alone cannot identify its cause.\')\n    para(\'The transfer candidates (frozen head, last block, and tuner winner) were compared by validation loss before test access. The selected candidate was <b>\'+selection[\'transfer_candidate\']+\'</b>. Thresholds and split hashes were saved in selection_before_test.json, then an exclusive test-start marker prevented a second final evaluation in this run directory.\')\n    shot(\'training_complete.png\',\'Actual completed experiment evidence and stage results.\',240)\n\n    page(\'2.5 - Held-out evaluation and decision threshold\')\n    para(\'AI is the positive class. A real photograph incorrectly called fake can damage trust, so the predeclared validation objective is 2 x false positives + false negatives. This 2:1 ratio is a product judgment, not a measured business cost. The algorithm checks every attainable validation boundary and chooses the higher threshold when costs tie. Test labels never influence this selection.\')\n    table([[\'Model\',\'Threshold\',\'Accuracy\',\'Precision\',\'Recall\',\'F1\',\'ROC-AUC\']]+[[m,f"{v[\'threshold\']:.5f}"]+[f"{v[k]:.4f}" for k in [\'accuracy\',\'precision\',\'recall\',\'f1\',\'roc_auc\']] for m,v in metrics.items()],[69,75,69,70,69,70,71],True)\n    table([[\'Model\',\'True real (TN)\',\'Real called AI (FP)\',\'AI missed (FN)\',\'True AI (TP)\']]+[[m,v[\'tn\'],v[\'fp\'],v[\'fn\'],v[\'tp\']] for m,v in metrics.items()],[80,98,119,98,98],True)\n    for m,v in metrics.items():\n        para(f"<b>{m}:</b> {v[\'fp\']} of 66 real images were called AI, and {v[\'fn\']} of 80 AI images were missed. Precision measures correctness of AI accusations; recall measures the fraction of AI images caught. ROC-AUC uses continuous scores and does not depend on the selected threshold.")\n    para(links[\'evaluation_interpretation\'])\n    para(\'The 146-image test set is small and comes from the same collection as training. These are point estimates without a new-source or new-generator guarantee. Source artifacts, label uncertainty, compression, and unseen generators limit generalization; the detector should support review rather than establish authenticity.\')\n    shot(\'evaluation.png\',\'Executed test metric comparison with the project context.\',260)\n\n    page(\'2.5 - Confusion matrices and ROC curves\')\n    for m in [\'baseline\',\'transfer\']:\n        a,b=T/f\'evaluation/{m}/confusion_matrix.png\',T/f\'evaluation/{m}/roc.png\'\n        cells=[]\n        for p in [a,b]:\n            iw,ih=ImageReader(str(p)).getSize();cells.append(Image(str(p),width=240,height=240*ih/iw))\n        story.append(Table([cells],colWidths=[246,247]))\n        para(m+\': confusion matrix (rows=true, columns=prediction) and held-out ROC curve.\',\'CaptionVG\')\n    para(\'Each final model predicts the test set exactly once. Probabilities, row-level predictions, and labels are saved, and the confusion matrix, ROC, metrics, and error grids are derived from that same pass. Accuracy is (TP+TN)/N; precision=TP/(TP+FP); recall=TP/(TP+FN); F1 is their harmonic mean. No model was retrained after the held-out metrics were exposed.\')\n\n    for m in [\'baseline\',\'transfer\']:\n        page(\'2.6 - Error analysis: \'+m)\n        image(T/f\'evaluation/{m}/misclassified.png\',f\'First ten available {m} errors in manifest order; true and predicted labels and AI scores are shown.\',235)\n        errs=rows(T/f\'evaluation/{m}/error_analysis.csv\')\n        assert all(r[\'visual_observation\'] and r[\'hypothesis_to_verify\'] for r in errs)\n        table([[\'Image\',\'Visible observation\',\'Hypothesis and check\']]+[[i+1,r[\'visual_observation\'],r[\'hypothesis_to_verify\']] for i,r in enumerate(errs)],[36,204,253],True)\n        para(\'Images are numbered left-to-right, top-to-bottom. Sorted paths place AI images first, so these first ten available errors are all false negatives; the confusion matrices cover both error types. These are visual observations and testable hypotheses, not proven causes. Proposed diagnostics were not used to retune the model or threshold.\')\n\n    page(\'2.7 - Keras Tuner: Fashion-MNIST toy\')\n    para(\'The toy uses a tiny flattened Fashion-MNIST classifier with one hidden ReLU layer and a 10-class softmax. A fixed stratified subset of 10,000 training and 2,000 validation images is taken from the built-in training partition; its test partition remains unused. Pixels are scaled to [0,1], with sparse categorical cross entropy for integer class labels.\')\n    para(\'Random search explores 32/64/128 hidden units and Adam learning rates 0.01/0.001/0.0001. Eight trials run up to five epochs with early stopping (patience 2). The objective is minimum validation loss, so lower is better.\')\n    toyrows=rows(T/\'tuner_toy/trials.csv\')\n    table([[\'Trial\',\'Status\',\'Units\',\'Learning rate\',\'Validation loss\']]+[[r[\'trial_id\'],r[\'status\'],r[\'units\'],r[\'learning_rate\'],f"{float(r[\'val_loss\']):.5f}"] for r in toyrows],[55,100,70,115,153])\n    best=js(T/\'tuner_toy/best_hyperparameters.json\')\n    para(\'Best toy settings: <b>\'+escape(str(best))+\'</b>. This lab demonstrates tuning mechanics on an unrelated built-in dataset; its score is not VisionGuard performance.\')\n    shot(\'tuner_toy.png\',\'Actual Fashion-MNIST toy search results.\',300)\n\n    page(\'2.7 - Applied VisionGuard tuning\')\n    para(\'Every trial starts from the same trained frozen-head checkpoint. Random search varies learning rate {1e-5, 3e-5, 1e-4}, head dropout {0.2, 0.4, 0.5}, and {1, 2, 4} unfrozen weighted layers within the final MobileNetV2 block/terminal convolution. Earlier layers and BatchNorm remain frozen. Each of eight trials runs up to five epochs with early stopping and learning-rate reduction.\')\n    trials=rows(T/\'tuner/trials.csv\')\n    assert sum(r[\'status\']==\'COMPLETED\' for r in trials)>=8\n    table([[\'Trial\',\'Status\',\'LR\',\'Dropout\',\'Unfrozen\',\'Val loss\']]+[[r[\'trial_id\'],r[\'status\'],r[\'learning_rate\'],r[\'dropout\'],r[\'unfrozen_layers\'],f"{float(r[\'val_loss\']):.5f}"] for r in trials],[40,100,80,75,75,123],True)\n    best=js(T/\'tuner/best_hyperparameters.json\')\n    para(\'Best applied settings: <b>\'+escape(str(best))+\'</b>. The actual winning checkpoint is retained; the final transfer candidate is selected against the head and standalone fine-tune candidates using validation loss.\')\n    ranges=js(T/\'tuner/descriptive_effect_ranges.json\')\n    winner=max(ranges,key=ranges.get)\n    table([[\'Parameter\',\'Range of marginal mean validation losses\']]+[[k,f\'{v:.6f}\'] for k,v in ranges.items()],[180,313])\n    para(\'<b>\'+winner+\'</b> has the widest observed marginal mean-loss range, and thus the strongest exploratory association in these trials. This is not causal importance: multiple settings change together, each value has few observations, and optimization is stochastic. A controlled one-factor comparison would be needed to isolate effects.\')\n    para(\'The marginal mean validation loss was 0.52172 at 1e-5 (2 trials), 0.51140 at 3e-5 (3), and 0.48805 at 1e-4 (3). Larger updates may have adapted the pretrained final layers faster within the short five-epoch budget. This is a hypothesis about the observed association, not proof that 1e-4 is universally best.\')\n    para(\'Choosing the minimum of many validation scores can select random favorable noise and overfit the validation split. The test set is consequently opened only after candidate and threshold selection. Further model changes after this report would require a fresh held-out set for an unbiased final check.\')\n    shot(\'tuner_applied.png\',\'Eight completed applied trials and winning settings.\',240)\n\n    page(\'Reproduction, troubleshooting, and references\')\n    para(\'The source repository contains the audit, pipeline/models, tuning, threshold/evaluation code, and standalone notebook. The evidence bundle includes the exact environment, split manifests, source hashes, curves, checkpoints, tuner state, and saved predictions. The report builder reads these outputs and refuses to build without COMPLETE.json and at least eight completed applied trials.\')\n    table([[\'Observed issue\',\'Resolution and evidence\'],[\'PNG decoding failure\',\'Lossless re-encoding from the audited original; 972/972 pixel hashes verified, one file repaired.\'],[\'Slow repeated full-resolution decode\',\'Cache resized deterministic inputs before per-epoch shuffle; augmentation remains dynamic.\'],[\'Google Drive storage quota\',\'Use runtime-local output storage and download evidence/checkpoints. Existing Drive content was not deleted.\']],[140,353])\n    para(\'For a new run: create a virtual environment; install requirements; audit the originals; verify labels and split hashes; run the PNG integrity repair; execute scripts.run_training with a new output directory. Use the provided unchanged settings for a pre-test resume. After test access, read saved predictions rather than repeating model selection.\')\n    para(\'Task 3 experiment tracking and Tasks 4-8 serving, database, containers, cloud deployment, and API integration are outside this Tasks 1-2 submission. The optional Grad-CAM bonus is also outside this report. Local saved experiment evidence is not claimed as remote MLflow tracking.\')\n    for label,url in [(\'Assignment dataset\',\'https://www.kaggle.com/datasets/cashbowman/ai-generated-images-vs-real-images\'),(\'TensorFlow transfer learning\',\'https://www.tensorflow.org/guide/keras/transfer_learning\'),(\'Keras Tuner getting started\',\'https://keras.io/keras_tuner/getting_started/\'),(\'DVC remote storage\',\'https://dvc.org/doc/user-guide/data-management/remote-storage\')]:\n        para(f\'<b>{label}:</b> <link href="{url}" color="#1c5c91">{url}</link>\',\'SmallVG\')\n    OUT.parent.mkdir(exist_ok=True)\n    doc=SimpleDocTemplate(str(OUT),pagesize=(595.276,841.89),rightMargin=51,leftMargin=51,topMargin=47,bottomMargin=56,title=\'VisionGuard - Tasks 1 and 2 - 23L-0727\',author=links.get(\'name\',\'Hayyan Haider\'))\n    doc.build(story,onFirstPage=footer,onLaterPages=footer)\n    print(OUT)\n\nif __name__==\'__main__\': build()\n', 'scripts/drop_invalid_pngs.py': '"""Exclude TensorFlow-undecodable PNG rows without deleting original images."""\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport json\nimport os\nimport shutil\nimport pandas as pd\nimport tensorflow as tf\n\n\ndef drop_invalid(processed, splits, report, previous_training=None):\n    processed, splits, report = map(Path, (processed, splits, report))\n    if previous_training and (Path(previous_training)/\'TEST_EVALUATION_STARTED.json\').exists():\n        raise RuntimeError(\'Test evaluation was already opened; do not change this test split.\')\n    frames, failures = {}, []\n    for split in [\'train\', \'val\', \'test\']:\n        frame = pd.read_csv(splits/f\'{split}.csv\')\n        keep = []\n        for rec in frame.itertuples():\n            try:\n                tf.io.decode_png(tf.io.read_file(str(processed/rec.path)), channels=3).numpy()\n                keep.append(True)\n            except (tf.errors.OpError, ValueError, OSError) as exc:\n                keep.append(False)\n                failures.append(dict(split=split, path=rec.path, label=rec.label, error=str(exc)))\n        frames[split] = frame.loc[keep].copy()\n        if set(frames[split].label) != {0, 1}:\n            raise RuntimeError(f\'{split} would lose an entire class; split lists left unchanged.\')\n        print(f\'{split}: kept {sum(keep)}, excluded {len(keep)-sum(keep)}\', flush=True)\n    stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')\n    backup = report/stamp\n    backup.mkdir(parents=True, exist_ok=False)\n    pd.DataFrame(failures, columns=[\'split\',\'path\',\'label\',\'error\']).to_csv(backup/\'excluded_images.csv\', index=False)\n    for split, frame in frames.items():\n        dest = splits/f\'{split}.csv\'\n        shutil.copy2(dest, backup/dest.name)\n        if failures:\n            temp = dest.with_suffix(\'.csv.tmp\')\n            frame.to_csv(temp, index=False)\n            os.replace(temp, dest)\n    result = dict(excluded=len(failures), backup=str(backup),\n                  splits={s:dict(files=len(f), ai=int(sum(f.label==1)), real=int(sum(f.label==0))) for s,f in frames.items()},\n                  note=\'File integrity exclusions before model test evaluation. Original files were not deleted.\')\n    summary_path = splits/\'split_summary.json\'\n    if failures and summary_path.exists():\n        shutil.copy2(summary_path, backup/summary_path.name)\n        summary = json.loads(summary_path.read_text())\n        summary[\'png_decode_filter\'] = result\n        summary_path.write_text(json.dumps(summary, indent=2))\n    (backup/\'filter_summary.json\').write_text(json.dumps(result, indent=2))\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n', 'scripts/dvc_demo.py': '"""Run an isolated two-version toy Git/DVC round trip; no user repo is modified."""\nimport argparse\nfrom datetime import datetime, timezone\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\n\n\ndef demo(destination):\n    dest = Path(destination).resolve()\n    if dest.exists():\n        raise FileExistsError(f\'Use a NEW directory; refusing to overwrite {dest}\')\n    repo, remote = dest/\'repo\', dest/\'remote\'\n    repo.mkdir(parents=True)\n    remote.mkdir()\n    transcript = []\n    def run(*args):\n        shown = \' \'.join(args)\n        print(f\'[{datetime.now(timezone.utc).isoformat()}] {repo.name}$ {shown}\', flush=True)\n        result = subprocess.run(args, cwd=repo, text=True, stdout=subprocess.PIPE,\n                                stderr=subprocess.STDOUT, check=False,\n                                env={**os.environ, \'DVC_NO_ANALYTICS\': \'true\',\n                                     \'DVC_SITE_CACHE_DIR\': str(dest/\'site-cache\'),\n                                     \'XDG_CONFIG_HOME\': str(dest/\'config\'),\n                                     \'XDG_CACHE_HOME\': str(dest/\'cache\')})\n        print(result.stdout, flush=True)\n        transcript.append(\'$ \'+shown+\'\\n\'+result.stdout)\n        (dest/\'transcript.txt\').write_text(\'\\n\'.join(transcript))\n        result.check_returncode()\n        return result.stdout.strip()\n    def git(*args):\n        return run(\'git\', \'-c\', \'user.name=VisionGuard Toy Demo\', \'-c\',\n                   \'user.email=visionguard-demo@example.invalid\', *args)\n    def dvc(*args):\n        return run(sys.executable, \'-m\', \'dvc\', *args)\n    git(\'init\', \'-b\', \'main\')\n    dvc(\'init\')\n    dvc(\'remote\', \'add\', \'-d\', \'local\', str(remote))\n    csv = repo/\'tiny.csv\'\n    v1 = \'id,label\\n1,real\\n2,ai\\n\'\n    csv.write_text(v1)\n    dvc(\'add\', \'tiny.csv\')\n    git(\'add\', \'.dvc\', \'.dvcignore\', \'.gitignore\', \'tiny.csv.dvc\')\n    git(\'commit\', \'-m\', \'Version 1: two labeled rows\')\n    first = git(\'rev-parse\', \'HEAD\')\n    dvc(\'push\')\n    csv.write_text(v1+\'3,real\\n\')\n    dvc(\'add\', \'tiny.csv\')\n    git(\'add\', \'tiny.csv.dvc\')\n    git(\'commit\', \'-m\', \'Version 2: add third row\')\n    second = git(\'rev-parse\', \'HEAD\')\n    dvc(\'push\')\n    git(\'log\', \'--oneline\', \'--all\')\n    git(\'checkout\', first)\n    dvc(\'checkout\')\n    assert csv.read_text() == v1\n    print(\'Restored older version:\\n\'+csv.read_text(), flush=True)\n    # Only this freshly created toy workspace/cache is removed, after both pushes.\n    csv.unlink()\n    shutil.rmtree(repo/\'.dvc\'/\'cache\')\n    dvc(\'pull\')\n    assert csv.read_text() == v1\n    print(\'PASS: recovered version 1 from remote after deleting local CSV AND cache.\', flush=True)\n    result = dict(first_commit=first, second_commit=second, restored_commit=first,\n                  remote=str(remote), csv=csv.read_text(), verified=True,\n                  note=\'Local round trip completed. Push both commits to your DagsHub toy repository separately.\')\n    (dest/\'result.json\').write_text(json.dumps(result, indent=2))\n    (dest/\'transcript.txt\').write_text(\'\\n\'.join(transcript))\n    return result\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--destination\', default=\'outputs/dvc_toy\')\n    print(json.dumps(demo(p.parse_args().destination), indent=2))\n', 'scripts/repair_processed.py': '"""Check every split PNG with TensorFlow and repair losslessly from audited sources."""\nimport argparse\nimport hashlib\nimport os\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\nimport tensorflow as tf\nfrom visionguard.data import rgb_image, file_hash\n\n\ndef pixel_hash(pixels):\n    h, w, channels = pixels.shape\n    if channels != 3:\n        raise ValueError(\'Expected three RGB channels\')\n    return hashlib.sha256(f\'{w}x{h}:RGB:\'.encode() + pixels.tobytes()).hexdigest()\n\n\ndef repair(raw, processed, splits, report):\n    raw, processed, splits, report = map(Path, (raw, processed, splits, report))\n    report.mkdir(parents=True, exist_ok=True)\n    changes, checked = [], 0\n    for split in [\'train\', \'val\', \'test\']:\n        frame = pd.read_csv(splits/f\'{split}.csv\')\n        for rec in frame.itertuples():\n            dest = processed/rec.path\n            reason = \'\'\n            try:\n                pixels = tf.io.decode_png(tf.io.read_file(str(dest)), channels=3).numpy()\n                if pixel_hash(pixels) != rec.pixel_sha256:\n                    raise ValueError(\'Processed pixels differ from audited pixel hash\')\n            except Exception as exc:\n                reason = str(exc)\n            if reason:\n                print(f\'Repairing {split}: {dest}\\nReason: {reason}\', flush=True)\n                source = raw/rec.source_path\n                if file_hash(source) != rec.sha256:\n                    raise ValueError(f\'Original file differs from audit: {source}\')\n                pixels = np.asarray(rgb_image(source), dtype=np.uint8)\n                if pixel_hash(pixels) != rec.pixel_sha256:\n                    raise ValueError(f\'Original decoded pixels differ from audit: {source}\')\n                # TensorFlow\'s encoder writes fresh image data without inherited ICC/EXIF.\n                encoded = tf.io.encode_png(pixels).numpy()\n                decoded = tf.io.decode_png(encoded, channels=3).numpy()\n                if pixel_hash(decoded) != rec.pixel_sha256:\n                    raise ValueError(f\'Re-encoded image failed lossless validation: {dest}\')\n                before = file_hash(dest) if dest.exists() else None\n                dest.parent.mkdir(parents=True, exist_ok=True)\n                temp = dest.with_name(dest.name+\'.repair.tmp\')\n                try:\n                    temp.write_bytes(encoded)\n                    os.replace(temp, dest)\n                finally:\n                    temp.unlink(missing_ok=True)\n                changes.append(dict(split=split, path=rec.path, source_path=rec.source_path,\n                                    reason=reason, before_sha256=before, after_sha256=file_hash(dest)))\n                (report/\'png_repairs.json\').write_text(json.dumps(changes, indent=2))\n            checked += 1\n            if checked % 100 == 0:\n                print(f\'Checked {checked} images; repaired {len(changes)}\', flush=True)\n    result = dict(checked=checked, repaired=len(changes), status=\'all split images TensorFlow-decodable and pixel hashes verified\')\n    (report/\'png_integrity.json\').write_text(json.dumps(result, indent=2))\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--raw\', required=True)\n    p.add_argument(\'--processed\', default=\'data/processed\')\n    p.add_argument(\'--splits\', default=\'outputs/data\')\n    p.add_argument(\'--report\', default=\'outputs/png_integrity\')\n    a = p.parse_args()\n    repair(a.raw, a.processed, a.splits, a.report)\n', 'scripts/run_training.py': '"""Train, tune, lock selection on validation, then evaluate each model on test once."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom tensorflow import keras\nfrom visionguard.training import (seed_everything, environment, dataset, baseline, transfer,\n    compile_model, unfreeze_last_block, fit_stage, augmented_batch, layer_evidence)\nfrom visionguard.tuning import toy, visionguard\nfrom visionguard.evaluation import choose_threshold, metrics, evaluate_predictions\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--root\', default=\'data/processed\')\n    parser.add_argument(\'--splits\', default=\'outputs/data\')\n    parser.add_argument(\'--out\', default=\'outputs/training\')\n    parser.add_argument(\'--size\', type=int, default=224)\n    parser.add_argument(\'--batch\', type=int, default=32)\n    parser.add_argument(\'--baseline-epochs\', type=int, default=20)\n    parser.add_argument(\'--head-epochs\', type=int, default=10)\n    parser.add_argument(\'--fine-epochs\', type=int, default=15)\n    parser.add_argument(\'--tuner-epochs\', type=int, default=5)\n    parser.add_argument(\'--seed\', type=int, default=42)\n    a = parser.parse_args()\n    out, splits = Path(a.out), Path(a.splits)\n    out.mkdir(parents=True, exist_ok=True)\n    # Never silently rerun a test evaluation or overwrite final test evidence.\n    marker = out / \'TEST_EVALUATION_STARTED.json\'\n    if marker.exists():\n        raise RuntimeError(\'Final test was already opened here. Read saved results; do not retune against this test.\')\n    config_path = out/\'config.json\'\n    current_config = vars(a)\n    if config_path.exists() and json.loads(config_path.read_text()) != current_config:\n        raise RuntimeError(\'Resume settings changed. Use the original settings or a new pre-test run directory.\')\n    split_identity = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()\n                      for p in (splits/\'train.csv\', splits/\'val.csv\', splits/\'test.csv\')}\n    identity_path = out/\'split_identity.json\'\n    if identity_path.exists() and json.loads(identity_path.read_text()) != split_identity:\n        raise RuntimeError(\'Split manifests changed; cannot reuse checkpoints or tuner state.\')\n    identity_path.write_text(json.dumps(split_identity, indent=2))\n    seed_everything(a.seed)\n    environment(out)\n    (out / \'config.json\').write_text(json.dumps(vars(a), indent=2))\n    train = dataset(splits/\'train.csv\', a.root, a.size, a.batch, True, a.seed)\n    val = dataset(splits/\'val.csv\', a.root, a.size, a.batch)\n    augmented_batch(train, out/\'augmented_batch.png\', a.seed)\n    stages = {}\n    def stage(name, builder, epochs):\n        path = out/name/\'best.keras\'\n        if path.exists() and (out/name/\'best_epoch.json\').exists():\n            return keras.models.load_model(path), json.loads((out/name/\'best_epoch.json\').read_text())\n        return fit_stage(builder(), train, val, out/name, epochs)\n    scratch, stages[\'baseline\'] = stage(\'baseline\', lambda: baseline(a.size, a.seed), a.baseline_epochs)\n    del scratch\n    keras.backend.clear_session()\n    seed_everything(a.seed)\n    head, stages[\'head\'] = stage(\'head\', lambda: transfer(a.size, seed=a.seed), a.head_epochs)\n    def fine_builder():\n        model = keras.models.load_model(out/\'head\'/\'best.keras\')\n        unfreeze_last_block(model)\n        return compile_model(model, 1e-5)\n    fine, stages[\'finetune\'] = stage(\'finetune\', fine_builder, a.fine_epochs)\n    del head, fine\n    keras.backend.clear_session()\n    toy(out/\'tuner_toy\', epochs=a.tuner_epochs, seed=a.seed)\n    tuned, hp = visionguard(train, val, out/\'head\'/\'best.keras\', out/\'tuner\',\n                           a.size, a.tuner_epochs, 8, a.seed)\n    layer_evidence(tuned, out/\'tuner\')\n    del tuned\n    keras.backend.clear_session()\n    # Compare transfer candidates on validation loss, never on held-out test.\n    candidates = {\'frozen_head\': out/\'head\'/\'best.keras\',\n                  \'last_block\': out/\'finetune\'/\'best.keras\', \'tuned\': out/\'tuner\'/\'best.keras\'}\n    scores = []\n    for name, path in candidates.items():\n        model = keras.models.load_model(path)\n        score = model.evaluate(val, verbose=0, return_dict=True)\n        scores.append(dict(candidate=name, path=str(path), **score))\n        del model\n    table = pd.DataFrame(scores).sort_values(\'loss\')\n    table.to_csv(out/\'transfer_validation_selection.csv\', index=False)\n    chosen = table.iloc[0]\n    final_models = {\'baseline\': out/\'baseline\'/\'best.keras\', \'transfer\': Path(chosen[\'path\'])}\n    thresholds = {}\n    val_y = pd.read_csv(splits/\'val.csv\').label.to_numpy()\n    for name, path in final_models.items():\n        model = keras.models.load_model(path)\n        p = model.predict(val, verbose=0).ravel()\n        t, choices = choose_threshold(val_y, p)\n        thresholds[name] = t\n        choices.to_csv(out/f\'{name}_validation_thresholds.csv\', index=False)\n        (out/f\'{name}_validation_metrics.json\').write_text(json.dumps(metrics(val_y, p, t), indent=2))\n        del model\n    selection = dict(transfer_candidate=chosen[\'candidate\'], thresholds=thresholds,\n        threshold_rule=\'Minimize 2*FP+FN on validation. AI=1. False accusation of a real photo costs twice a missed AI image.\',\n        stages=stages, tuned_hyperparameters=hp,\n        split_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in splits.glob(\'*.csv\')})\n    (out/\'selection_before_test.json\').write_text(json.dumps(selection, indent=2))\n    # Exclusive create marks the test as opened even if a later plot/save fails.\n    with marker.open(\'x\') as f:\n        json.dump(selection, f, indent=2)\n    test_frame = pd.read_csv(splits/\'test.csv\')\n    test = dataset(splits/\'test.csv\', a.root, a.size, a.batch)\n    results = []\n    for name, path in final_models.items():\n        model = keras.models.load_model(path)\n        # Exactly one test inference pass per final model; reuse predictions for all figures.\n        p = model.predict(test, verbose=0).ravel()\n        dest = out/\'evaluation\'/name\n        dest.mkdir(parents=True, exist_ok=True)\n        np.save(dest/\'test_probabilities.npy\', p)\n        result = evaluate_predictions(test_frame, p, thresholds[name], a.root, dest)\n        results.append(dict(model=name, **result))\n        del model\n    comparison = pd.DataFrame(results)\n    comparison.to_csv(out/\'comparison.csv\', index=False)\n    print(comparison.to_string(index=False))\n    (out/\'COMPLETE.json\').write_text(json.dumps(dict(status=\'complete\', models=list(final_models)), indent=2))\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/version_dataset.py': '"""Apply real dataset DVC versioning in Colab; keep bytes and Git backup on Drive."""\nimport argparse\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport os\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--raw\', required=True)\n    p.add_argument(\'--splits\', required=True)\n    p.add_argument(\'--persistent\', required=True)\n    p.add_argument(\'--git-name\', required=True)\n    p.add_argument(\'--git-email\', required=True)\n    a = p.parse_args()\n    root, persistent = Path.cwd(), Path(a.persistent).resolve()\n    persistent.mkdir(parents=True, exist_ok=True)\n    if (root/\'.git\').exists():\n        raise RuntimeError(\'Repository already exists. Use docs/DVC_GUIDE.md to inspect and finish versioning safely.\')\n    env = {**os.environ, \'DVC_NO_ANALYTICS\': \'true\',\n           \'DVC_SITE_CACHE_DIR\': str(root/\'outputs\'/\'dvc-site-cache\')}\n    def run(*args):\n        print(\'$ \'+\' \'.join(map(str, args)), flush=True)\n        subprocess.run(list(map(str, args)), check=True, env=env)\n    run(\'git\', \'init\', \'-b\', \'main\')\n    run(\'git\', \'config\', \'user.name\', a.git_name)\n    run(\'git\', \'config\', \'user.email\', a.git_email)\n    run(sys.executable, \'-m\', \'dvc\', \'init\')\n    # Persist this writable cache location so subsequent dvc commands work locally too.\n    run(sys.executable, \'-m\', \'dvc\', \'config\', \'--local\', \'core.site_cache_dir\', root/\'outputs\'/\'dvc-site-cache\')\n    raw_dest = root/\'data\'/\'raw\'\n    if Path(a.raw).resolve() != raw_dest.resolve():\n        if raw_dest.exists():\n            raise FileExistsError(\'data/raw already exists; inspect rather than overwrite it\')\n        shutil.copytree(a.raw, raw_dest)\n    if not (root/\'data\'/\'processed\').is_dir():\n        raise FileNotFoundError(\'Run the audit/preparation cell first to create data/processed\')\n    split_dest = root/\'data\'/\'splits\'\n    split_dest.mkdir(parents=True, exist_ok=True)\n    for name in [\'train.csv\', \'val.csv\', \'test.csv\', \'class_mapping.json\', \'split_summary.json\']:\n        shutil.copy2(Path(a.splits)/name, split_dest/name)\n    (root/\'data\'/\'.gitignore\').write_text(\'/raw\\n/processed\\n/splits\\n\')\n    run(sys.executable, \'-m\', \'dvc\', \'add\', \'data/raw\', \'data/processed\', \'data/splits\')\n    run(\'git\', \'add\', \'data/raw.dvc\', \'data/processed.dvc\', \'data/splits.dvc\', \'data/.gitignore\')\n    storage = persistent/\'dvc_remote\'\n    storage.mkdir(exist_ok=True)\n    run(sys.executable, \'-m\', \'dvc\', \'remote\', \'add\', \'-d\', \'storage\', storage)\n    run(sys.executable, \'-m\', \'dvc\', \'push\')\n    run(sys.executable, \'-m\', \'dvc\', \'status\', \'-c\')\n    run(\'git\', \'add\', \'.gitignore\', \'.dvc\', \'.dvcignore\', \'data/raw.dvc\', \'data/processed.dvc\',\n        \'data/splits.dvc\', \'data/.gitignore\', \'visionguard\', \'scripts\', \'docs\', \'requirements.txt\', \'README.md\')\n    run(\'git\', \'commit\', \'-m\', \'Task 1: version original and processed dataset plus audited split lists\')\n    # A bare Git backup keeps the committed metadata after the Colab runtime is recycled.\n    backup = persistent/\'git_backup.git\'\n    if backup.exists():\n        raise FileExistsError(\'Git backup already exists; inspect it before changing it\')\n    run(\'git\', \'clone\', \'--bare\', root, backup)\n    run(\'git\', \'rev-parse\', \'HEAD\')\n    print(\'DVC dataset push and persistent Git backup complete. Publish commits to DagsHub per DVC_GUIDE.md.\', flush=True)\n\nif __name__ == \'__main__\':\n    main()\n', 'docs/DVC_GUIDE.md': "# Task 1.4 — DVC and DagsHub\n\n## Toy lab already scripted\n\nRun `python -m scripts.dvc_demo --destination outputs/dvc_toy`. The destination must be new. The script prints every command and creates:\n\n- `repo/`: an isolated Git+DVC repository with two committed CSV versions.\n- `remote/`: actual persistent DVC objects for both versions.\n- `transcript.txt`: command log.\n- `result.json`: verified commit hashes and recovered content.\n\nIt checks out version 1, removes the toy CSV AND the toy cache, then pulls from the remote and verifies the exact CSV content. Clearing the toy cache proves retrieval from remote, rather than a local cache checkout. It never deletes any real dataset.\n\nTo put BOTH toy commits on DagsHub, create a separate empty DagsHub repository such as `<RollNumber>_A2_Ops_DVC_Toy`, then run from the toy's `repo` directory:\n\n```bash\ngit remote add origin https://dagshub.com/YOUR_USERNAME/YOUR_TOY_REPOSITORY.git\ngit push -u origin main\n```\n\nThe toy ends on the old commit in detached HEAD, but `main` still references version 2, so pushing `main` publishes both commits. Authenticate through a credential helper or a private prompt. Do not embed tokens in remote URLs or saved notebook cells. Capture DagsHub's commit view plus the terminal output. The local toy author is explicitly a demo identity; use your own Git identity for the actual project.\n\n## Apply DVC to the actual dataset\n\nCreate GitHub and DagsHub repositories named exactly `<RollNumber>_A2_Ops`. Connect/import the GitHub repository in DagsHub if desired. Use your real roll number; the assistant has not created external accounts or repos.\n\nInside the actual project folder (in Colab, `WORK_DIR`), use the following commands once. In notebook Python cells use `subprocess.run([...], check=True, cwd=WORK_DIR)` or run these as a `%%bash` cell with `cd` to the project folder. Replace placeholder paths with the printed actual paths; `DATA_ROOT`, `PROJECT_DIR`, etc. are Python variables, not automatic shell variables.\n\n```bash\ngit init -b main\ngit config user.name 'YOUR NAME'\ngit config user.email 'YOUR GIT EMAIL'\ndvc init\n```\n\nCopy the downloaded original dataset to `data/raw` while preserving its class folders. This may require substantial extra disk space. Keep `data/processed` from the audit. Copy `train.csv`, `val.csv`, `test.csv`, `class_mapping.json`, `split_summary.json` from the audit evidence folder into `data/splits/`. Use this Python cell in Colab:\n\n```python\nimport shutil\nraw_copy = WORK_DIR / 'data/raw'\nif not raw_copy.exists():\n    shutil.copytree(DATA_ROOT, raw_copy)\nsplit_copy = WORK_DIR / 'data/splits'\nsplit_copy.mkdir(parents=True, exist_ok=True)\nfor name in ['train.csv', 'val.csv', 'test.csv', 'class_mapping.json', 'split_summary.json']:\n    shutil.copy2(DATA_EVIDENCE / name, split_copy / name)\n```\n\nTrack the immutable source, derived inputs, and split identity:\n\n```bash\ndvc add data/raw data/processed data/splits\ngit add data/*.dvc\n```\n\n`dvc add` creates `.dvc` pointer files and ignore rules if needed. Git must include the pointers and code, while ignoring the large image directories. This project's ignore rules ignore only the three image/manifest directories, allowing `data/*.dvc` to remain visible. Stage the pointers before pushing so DVC discovers them reliably.\n\n### Option A — persistent local-folder remote (allowed in assignment)\n\nIn Colab use a folder on your mounted Google Drive, for example the following Python cell:\n\n```python\nremote_path = PROJECT_DIR / 'dvc_remote'\nremote_path.mkdir(parents=True, exist_ok=True)\nsubprocess.run(['dvc', 'remote', 'add', '-d', 'storage', str(remote_path)], cwd=WORK_DIR, check=True)\nsubprocess.run(['dvc', 'push'], cwd=WORK_DIR, check=True)\nsubprocess.run(['dvc', 'status', '-c'], cwd=WORK_DIR, check=True)\n```\n\nA remote under `/content` alone disappears with the runtime; Drive storage persists. Drive quota must accommodate raw and processed images plus DVC objects. This remote path is specific to your mounted filesystem; another user must mount/access the shared remote or reconfigure their own location.\n\n### Option B — DagsHub object storage\n\nInstall `dvc[s3]` (included in Colab setup), then copy current commands from your repository's **Remote → Data → DVC** panel. The documented pattern is:\n\n```bash\ndvc remote add -d storage s3://dvc\ndvc remote modify storage endpointurl https://dagshub.com/YOUR_USERNAME/YOUR_PROJECT.s3\n```\n\nSet credentials privately from a Python cell (input is hidden):\n\n```python\nfrom getpass import getpass\ntoken = getpass('DagsHub token: ')\nfor field in ['access_key_id', 'secret_access_key']:\n    subprocess.run(['dvc', 'remote', 'modify', '--local', 'storage', field, token],\n                   cwd=WORK_DIR, check=True, stdout=subprocess.DEVNULL)\ndel token\nsubprocess.run(['dvc', 'push'], cwd=WORK_DIR, check=True)\n```\n\n`--local` stores secrets in ignored `.dvc/config.local`. Never print it or include it in screenshots/ZIPs. Consult the repo's current remote panel if the endpoint or credential method differs.\n\n### Commit and publish pointers and source\n\n```bash\ngit add .gitignore .dvc .dvcignore data/*.dvc visionguard scripts docs requirements.txt README.md\ngit commit -m 'Task 1: version audited source, processed images and split lists'\ngit remote add origin https://github.com/YOUR_USERNAME/YOUR_PROJECT.git\ngit remote add dagshub https://dagshub.com/YOUR_USERNAME/YOUR_PROJECT.git\ngit push -u origin main\ngit push dagshub main\n```\n\nRun a remote command only once; if it already exists, inspect `git remote -v` and use `git remote set-url`. If DagsHub is connected as an automatic mirror, use its sync mechanism instead of pushing to a read-only mirror. Add the completed notebook and selected report evidence deliberately; the full model/dataset archives do not belong in Git. Preserve the repo metadata by pushing it before the Colab runtime expires.\n\nFor recovery in a new clone: configure remote access, then `dvc pull`. For a prior version: `git checkout <commit> -- data/raw.dvc data/processed.dvc data/splits.dvc`, then `dvc checkout` (or `dvc pull` if not cached). This changes tracked data to that snapshot; keep uncommitted work safe first. Demonstrate destructive deletion/recovery only in the isolated toy.\n\n## Explain in your report\n\nA `.dvc` file is a small text manifest with the data path, checksum and metadata, typically size and file count for directories. The directory checksum references a DVC manifest of its contents. DVC's checksum/object identity is not necessarily SHA-256; the separate audit explicitly computes SHA-256. Git versions the small pointer and code; DVC caches and transfers the large bytes. Git history retains previous versions, so committing large image datasets directly bloats clones and history without useful text diffs.\n\nExplain `git init`, `dvc init`, `dvc add`, `git commit`, `dvc remote add`, `dvc push`, `git checkout`, `dvc checkout` and `dvc pull` in your own words, tied to your actual outputs.\n\nReferences: [DVC remote storage](https://dvc.org/doc/user-guide/data-management/remote-storage), [DagsHub DVC integration](https://dagshub.com/docs/integration_guide/dvc/).\n", 'docs/LOCAL_SETUP.md': '# Local setup and execution\n\nUse Python 3.11 or 3.12 for broad TensorFlow wheel availability. A separate environment is required; do not install into system Python.\n\n```bash\npython3.11 -m venv .venv\nsource .venv/bin/activate\npython -m pip install --upgrade pip\npython -m pip install -r requirements.txt\npython -c "from visionguard.training import environment; environment(\'outputs/environment\')"\n```\n\n`venv` isolates packages. `pip install` installs dependencies. `environment` records Python, TensorFlow, Keras, GPU devices and a complete exact-version pip freeze. Dependency ranges bootstrap installation; `requirements-lock.txt` captures the actual run. TensorFlow CPU is sufficient for smoke checks but full training is better on Colab GPU. On Colab, keep its TensorFlow installation rather than replacing the CUDA stack.\n\nDownload/unzip the exact Kaggle dataset into `data/raw` (or use the Colab downloader). Inspect actual folder names; change the JSON mapping accordingly:\n\n```bash\npython -m visionguard.data --root data/raw --processed data/processed --out outputs/data --class-map \'{"AiArtData":"ai","RealArt":"real"}\'\npython -m scripts.dvc_demo --destination outputs/dvc_toy\npython -m scripts.run_training --root data/processed --splits outputs/data --out outputs/training\npython -m unittest discover -s tests -v\n```\n\nThe auditor raises an error for an image with an unrecognized label folder. It does not silently label unknown folders or silently ignore decoding failures. Nonimage sidecars are listed separately; corruption totals cover candidate image files.\n\nIf decoding/augmentation memory is tight, set `--batch 16`. Do not reduce trial count below eight for the graded run. Changing hyperparameters invalidates resume state: use a separate run directory BEFORE inspecting any test results. A fixed validation split can itself be overfit through many trials. Once test has been opened, keep it held out from further model decisions and report the results honestly.\n\nNotebook rebuild after editing source/docs:\n\n```bash\npython scripts/build_notebook.py\n```\n\nThe generated notebook embeds source so it runs independently of a Git repository. Rebuild whenever source changes.\n', 'docs/REPORT_WORKSHEET.md': "# Tasks 1 and 2 report worksheet\n\nThis is a scaffold, not a completed experiment report. Replace every prompt with observed outputs and your explanation. Do not claim a command ran remotely or a model achieved a score until you have evidence. Use your own screenshots; code blocks alone are not evidence of execution.\n\nFront matter: name, roll number, project name `<RollNumber>_A2_Ops`, GitHub and DagsHub links, graded commit hash, scope “Tasks 1 and 2”, and AI-assistance disclosure. Add deployment URLs only when later tasks are done. Follow your instructor's PDF page limit.\n\nFor screenshots include readable project name plus username/date/time or browser context. Caption and reference every screenshot/figure in prose. Explain what you did, why, how, and what went wrong under each sub-task ID.\n\n| ID | Evidence produced / action still required | Explain in your own words |\n|---|---|---|\n| 1.1 | `environment/environment.json`, `requirements-lock.txt`; screenshot versions and GPU output; local venv commands in setup guide | Isolation, exact versions, GPU detection and actual runtime |\n| 1.2 | `data_evidence/audit.csv`, `class_counts.csv`, `summary_statistics.csv`, `audit_summary.json`, `corrupt.csv`, `exact_duplicates.csv`, `pixel_duplicates.csv`, `label_conflicts.csv` | Original-file walk, SHA-256, unreadability, byte vs pixel duplicates, raw counts vs retained split counts |\n| 1.3 | `samples.png`, `width_histogram.png`, `height_histogram.png`, `size_bytes_histogram.png` | Each figure needs a caption and 2–3 sentences grounded in actual observations |\n| 1.4 | Toy `transcript.txt`, `result.json`, `.dvc` pointer; actual dataset pointers; DVC push/status; DagsHub two toy commits and actual data commit screenshots | Git vs DVC storage, version change/rollback, remote recovery, pointer fields, why large bytes stay out of Git |\n| 2.1 | Split manifests, `split_summary.json`, `training/augmented_batch.png` | Every tf.data stage, 224px compute/detail tradeoff, aspect ratio and borders, augmentation and leakage controls |\n| 2.2 | `training/baseline/model_summary.txt`, `layers.csv` (include table in report), screenshot model summary | Purpose of each layer, sigmoid/P(AI), binary cross entropy, Adam, metrics and regularization |\n| 2.3 | `head/curves.png`, `finetune/curves.png`, each `trainable_layers.json`; `transfer_validation_selection.csv` | ImageNet transfer, [-1,1] preprocessing, head stage, last block, frozen BatchNorm, recompile, 100× smaller LR |\n| 2.4 | All main-stage `curves.png`, `best_epoch.json`, `epochs.csv`, `best.keras`, `config.json` | Seed, determinism limits, early stopping, minimum validation-loss epoch, checkpointing and LR reduction |\n| 2.5 | `training/comparison.csv`, each `evaluation/<model>/metrics.json`, `confusion_matrix.png`, `roc.png`, validation threshold CSVs, `selection_before_test.json` | All six requested metrics, positive label, threshold cost assumption and actual selected thresholds, one test pass per model |\n| 2.6 | Each model's `misclassified.png`, `error_analysis.csv` | Fill ten image-specific observations/hypotheses; if fewer errors exist, state that honestly |\n| 2.7 | `tuner_toy/trials.csv`, `tuner/trials.csv`, `best_hyperparameters.json`, three `*_association.csv` files | Random search, search spaces, ≥8 completed applied trials, which setting appeared to matter, uncertainty/confounding and validation overfit |\n\n## EDA writing prompts — do not treat these as observations\n\nFor the sample grid, describe subjects and visible variation per class. Identify any visible source artifact only after looking at the actual images, and state whether labels look plausible. Explain why a few samples cannot establish the dataset's full representativeness.\n\nFor each width/height/size histogram, describe the central range, spread and outliers using the actual table/figure. Compare class distributions and note the logarithmic x-axis. Explain how source resolution, compression or borders might become shortcuts rather than evidence of generation.\n\n## Model rationale to adapt\n\nThe 224×224 choice is compatible with MobileNetV2 pretrained inputs and limits GPU memory. It can erase high-frequency forensic details. Letterboxing avoids stretching subjects but may introduce class-correlated borders; this is a limitation to investigate.\n\nA convolution learns local features with shared weights. BatchNorm stabilizes activations. ReLU adds nonlinear behavior. Pooling reduces spatial size and computation. Global average pooling avoids a large flatten/dense stack; a small dense layer combines learned channels. Dropout regularizes, and a sigmoid provides a binary score. Scores need not be calibrated probabilities of authenticity.\n\nPretrained convolution features provide a useful starting point, but ImageNet classification and AI-image detection are different tasks. Frozen-head training preserves pretrained features initially. Last-block tuning allows adaptation at a small learning rate while protecting earlier features. Model selection and all thresholds must be locked before test inference.\n\n## Evaluation discussion to complete after the run\n\nRecord the selected transfer candidate, each validation threshold, each best epoch, and the actual test metric table. Describe the confusion matrix in terms of real photos falsely accused and AI images missed. The default product judgment penalizes false accusations 2×; justify it for this use case and acknowledge it is an assumption. Discuss tradeoffs in precision/recall rather than selecting a winner from accuracy alone. If you name a preferred model, distinguish the pre-test validation choice from the retrospective test comparison.\n\n## Error worksheet\n\nFill `visual_observation` with a visible fact (for example, “strong JPEG blocks around edges” only if actually present). Fill `hypothesis_to_verify` with a possible explanation and a check (“resize/compression may suppress artifacts; test on a separately held-out compression sweep”). A hypothesis is not a proven cause. Do not label an unseen image as blurry, watermarked or painting-like.\n\n## Tuner discussion\n\nRecord the toy's best units/LR, all eight applied completed trials, and winning LR/dropout/unfrozen-layer count. Compare marginal means and individual trials. The widest marginal mean-loss range is an exploratory association, not reliable causal importance: trials vary multiple settings, the sample is small and training is stochastic. Explain that picking the best of many validation scores can select noise, making validation optimistic. A final untouched test checks generalization only once; further model iteration would require fresh holdout data.\n\n## What went wrong\n\nRecord actual issues: authentication, folder mapping, corrupt inputs, GPU quota, out-of-memory, interrupted training, or unstable fine-tuning only if they occurred. Explain your fix and evidence. Do not invent a troubleshooting story to fill the section.\n\n## Submission checks\n\nVerify class labels and zero overlap of retained pixel hashes across splits. Verify eight applied tuner trials have `COMPLETED` status. Check all figures are readable, both transfer stages appear, ten available errors have individual hypotheses, and exact environment versions are included. Verify your DagsHub commits/storage and that tokens are absent from repository history and screenshots. Disclose AI assistance accurately. Export your authored report to the required `<RollNumber>_VisionGuard.pdf` filename once real evidence is inserted.\n", 'docs/VERIFICATION.md': "# Completed real-data verification - 6 November 2026\n\nThe actual Colab T4 experiment is complete, beyond the earlier synthetic software checks.\n\n- 973 original readable images; zero unreadable candidates; one duplicate removed; 972 retained.\n- Stratified split sizes: 680/146/146. Original membership is identical in local and Colab manifests.\n- Zero decoded-pixel hash overlap between splits.\n- One Colab processed PNG repaired losslessly from its audited original; 972/972 pixel hashes verified.\n- Scratch CNN: 10 epochs, best epoch 6. Frozen head: 10 epochs, best 10. Last-block stage: 15 epochs, best 11.\n- Eight completed Fashion-MNIST toy trials and eight completed applied trials.\n- Tuned transfer selected by validation loss; thresholds locked before test.\n- Saved probabilities reproduce all reported test metrics without another model inference pass.\n- Ten displayed errors per model inspected and annotated with visible observations/testable hypotheses.\n- Local Git/DVC two-version push/delete-cache/pull recovery passed. Both toy commits are published on DagsHub's dvc-toy branch.\n- Real raw/processed/split DVC objects pushed to a local remote and verified in sync.\n- Four local unit tests passed after the input caching change.\n\nFull evidence lives under outputs/23L-0727_A2_Ops. FINAL_VERIFICATION.json records the completed evidence checks. The report and submission archive are under submission/.\n\nNo remote MLflow runs, API deployment, Tasks 3-8, or Grad-CAM bonus are claimed.\n", 'requirements.txt': '# Colab supplies TensorFlow; its exact runtime versions are captured before training.\ntensorflow>=2.16,<2.22\nkeras-tuner>=1.4.7,<1.5\ntensorboard>=2.16,<2.22\nkagglehub>=0.3,<1\nPillow>=10,<13\nnumpy>=1.26,<3\npandas>=2.2,<3.1\nmatplotlib>=3.8,<4\nscikit-learn>=1.4,<2\ndvc[s3]>=3.50,<4\nnbformat>=5.10,<6\n', '.gitignore': '.venv/\n__pycache__/\n*.pyc\n.env\n.env.*\n!.env.example\nkaggle.json\n*.keras\n*.h5\n*.zip\ndata/raw/\ndata/processed/\ndata/splits/\noutputs/\ndvc_toy/\ndvc_remote/\n.ipynb_checkpoints/\n.venv-tf/\nvisionguard_tasks_1_2.zip\n\n.DS_Store\n', 'README.md': "# VisionGuard — Assignment Tasks 1 and 2\n\nStart with **[notebooks/VisionGuard_Tasks_1_2.ipynb](notebooks/VisionGuard_Tasks_1_2.ipynb)**. It is self-contained: upload only this notebook to [Google Colab](https://colab.research.google.com/), select a GPU, enter your roll number, and run in order. No source ZIP upload is needed. Confirm the actual dataset folder names before assigning labels.\n\nThe notebook downloads the assignment's Kaggle dataset, audits it, produces EDA, creates deduplicated 70/15/15 splits, trains a scratch CNN and a two-stage MobileNetV2, runs the Fashion-MNIST tuner lab and eight VisionGuard trials, selects thresholds on validation, then evaluates the two final models on test once. Evidence and checkpoints persist to Google Drive when mounting succeeds. If mounting fails, the notebook continues on local runtime storage; download results before runtime deletion. Set `USE_GOOGLE_DRIVE = False` to skip mounting. It includes a runnable DVC toy and instructions to version the actual dataset and push commits to DagsHub.\n\n## Files\n\n- `visionguard/data.py`: original-file metadata/SHA-256 audit, corruption and duplicate detection, figures and split manifests.\n- `visionguard/training.py`: tf.data, training-only augmentation, models, callbacks and layer evidence.\n- `visionguard/tuning.py`: toy and applied Keras Tuner random searches.\n- `visionguard/evaluation.py`: validation cost threshold, metrics, confusion matrix, ROC and error grids.\n- `scripts/run_training.py`: main orchestration and single-use test guard.\n- `scripts/repair_processed.py`: verifies TensorFlow PNG decoding/pixel hashes and losslessly repairs incompatible or partial files from audited originals.\n- `scripts/version_dataset.py`: applies dataset DVC tracking, uploads objects and backs up Git on persistent storage.\n- `scripts/dvc_demo.py`: real two-commit Git/DVC push/delete/cache-clear/pull verification.\n- `docs/DVC_GUIDE.md`: dataset versioning and DagsHub publishing steps.\n- `docs/REPORT_WORKSHEET.md`: exact evidence mapping and writing prompts.\n- `docs/LOCAL_SETUP.md`: isolated Python setup and CLI commands.\n- `tests/test_data.py`: synthetic-fixture checks for corruption, leakage and threshold optimization.\n\n## Completed Tasks 1 and 2 - 6 November 2026\n\nThe full assignment dataset was audited and the actual experiment ran on Colab's Tesla T4 (Python 3.13.16, TensorFlow 2.21.0, Keras 3.13.2). There are 973 readable original images (538 AI, 435 real), zero unreadable candidates, and one duplicate removed. The disjoint stratified splits contain 680 train, 146 validation, and 146 test images. One incompatible processed PNG was repaired losslessly and all 972 retained pixel hashes were verified.\n\n| Final model | Accuracy | Precision | Recall | F1 | ROC-AUC | Validation threshold |\n|---|---:|---:|---:|---:|---:|---:|\n| Scratch CNN | 0.4932 | 0.6667 | 0.1500 | 0.2449 | 0.6672 | 0.651028 |\n| Tuned MobileNetV2 | 0.7055 | 0.7761 | 0.6500 | 0.7075 | 0.8377 | 0.680687 |\n\nEight Fashion-MNIST toy trials and eight applied trials completed. The applied winner used LR 0.0001, dropout 0.2, and two unfrozen weighted layers. Validation loss chose the tuned transfer model before test access. Each final model made one held-out inference pass; all figures and metrics reuse the saved predictions. Ten errors per model have image-specific observations and hypotheses in the evidence bundle.\n\n- GitHub: https://github.com/Muzxa/23L-0727_A2_Ops\n- DagsHub: https://dagshub.com/Muzxa/23L-0727_A2_Ops\n- DVC toy commits: the `dvc-toy` branch on both platforms.\n- Report: `submission/23L-0727_VisionGuard.pdf`.\n- Local full evidence: `outputs/23L-0727_A2_Ops/colab/`.\n- Local DVC object storage: `outputs/23L-0727_A2_Ops/versioned_storage/dvc_remote/`.\n\nThe DVC storage is a real local remote, an option permitted by Task 1.4. Data bytes are not uploaded to DagsHub Storage; DagsHub mirrors the Git source, pointers, and toy commits. A new machine needs the local remote or must recreate the dataset from Kaggle. The original dataset has artwork in its real folder, so labels must not be presented as verified photographic authenticity. Near-duplicate and common-source leakage remain limitations.\n\nGoogle Drive quota prevented notebook saves. The completed run used runtime-local outputs, which were downloaded and verified locally. Deterministic resized inputs are cached before per-epoch shuffling to avoid repeated full-resolution decoding. Augmentation remains inside the model and is never cached.\n\nTasks 3-8, cloud/API deployment, remote MLflow tracking, and the optional Grad-CAM bonus are outside this Tasks 1-2 submission. Saved local experiments are not claimed as MLflow runs.\n\n## Design choices\n\nAI=1, real=0. Training inputs are RGB [0,255]; preprocessing is embedded in the saved model. Resize with padding preserves aspect ratio. Validation/test are never augmented. Byte hashes audit the source files; decoded-pixel hashes detect identical images saved with different metadata. Conflicting labels for identical pixels are excluded. Near-duplicate/source leakage remains a limitation; no source IDs are supplied here. Existing dataset folder splits are deliberately replaced with a global deduplicated split, so results should not be compared as if they used an official benchmark split.\n\nThe primary transfer experiment freezes the head's backbone, then unfreezes only the last MobileNetV2 block plus terminal convolution. Tuning varies the number of weighted layers within that region. BatchNorm statistics remain frozen. Transfer candidate selection uses validation loss; threshold selection minimizes the predeclared validation cost `2*FP+FN`. Real→AI is assumed twice as costly because false accusations can damage trust. This is an explicit product judgment, not a measured business fact.\n\nThe saved test guard blocks a second final evaluation in the same run directory. If interrupted after test probabilities are saved, reconstruct figures from them; do not train a new model against the exposed test. Resume only with unchanged data/settings. Checkpoints are not a substitute for a full experiment tracker.\n\n## Authoritative references\n\n- [Assignment dataset on Kaggle](https://www.kaggle.com/datasets/cashbowman/ai-generated-images-vs-real-images)\n- [TensorFlow transfer learning guide](https://www.tensorflow.org/guide/keras/transfer_learning)\n- [Keras Tuner getting started](https://keras.io/keras_tuner/getting_started/)\n- [DVC remote storage](https://dvc.org/doc/user-guide/data-management/remote-storage)\n- [DagsHub DVC integration](https://dagshub.com/docs/integration_guide/dvc/)\n\nAI-assistance disclosure: “OpenAI Codex helped prepare and repair the code, operated the Colab experiment, inspected outputs, and drafted the report. I am responsible for reviewing and understanding the submitted work.”\n"}
for relative, contents in BUNDLED_FILES.items():
    path = WORK_DIR / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents)
# Persist the source alongside evidence without persisting large decoded images.
for relative, contents in BUNDLED_FILES.items():
    path = PROJECT_DIR / 'source' / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents)
print('Wrote', len(BUNDLED_FILES), 'source files')

In [ ]:
from visionguard.training import environment, seed_everything
seed_everything(SEED)
environment(PROJECT_DIR / 'environment')
from datetime import datetime, timezone
print('Project:', PROJECT_NAME, 'UTC:', datetime.now(timezone.utc).isoformat())

## Download the assignment dataset and inspect class folders
Uses the exact Kaggle dataset in the assignment. If authentication is required, run `kagglehub.login()` privately and retry. Do not paste a Kaggle token into a saved cell. If access or terms must be accepted, do that on Kaggle first.

Check the printed directory names. The mapping below is explicit: change **keys** to the actual folder names containing images, while keeping values `ai` and `real`. Do not guess labels from filenames. The nearest mapped ancestor determines the class.

In [ ]:
import kagglehub
DATA_ROOT = Path(kagglehub.dataset_download('cashbowman/ai-generated-images-vs-real-images'))
print('Dataset root:', DATA_ROOT)
from collections import Counter
folder_counts = Counter(str(p.parent.relative_to(DATA_ROOT)) for p in DATA_ROOT.rglob('*') if p.is_file())
for folder, count in folder_counts.most_common(40):
    print(f'{count:7d} files  {folder}')

In [ ]:
# EDIT these keys if Kaggle uses other folder names; verify visually after audit.
CLASS_FOLDERS = {'AiArtData': 'ai', 'RealArt': 'real'}
assert set(CLASS_FOLDERS.values()) == {'ai', 'real'}
CLASS_MAP_JSON = __import__('json').dumps(CLASS_FOLDERS)
DATA_EVIDENCE = PROJECT_DIR / 'data_evidence'
PROCESSED = WORK_DIR / 'data' / 'processed'
import subprocess, sys
def run_module(module, *args):
    from collections import deque
    tail = deque(maxlen=40)
    command = [sys.executable, '-u', '-m', module, *map(str, args)]
    with subprocess.Popen(command, cwd=WORK_DIR, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            tail.append(line)
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f'{module} exited with code {returncode}. Last output:\n' + ''.join(tail))
run_module('visionguard.data', '--root', DATA_ROOT, '--out', DATA_EVIDENCE,
           '--processed', PROCESSED, '--class-map', CLASS_MAP_JSON, '--seed', SEED)

## 1.2–1.3 — Audit and EDA evidence
The audit describes **original** files. Training copies are lossless RGB PNGs with EXIF orientation corrected, and animated images use their first frame. Exact byte duplicates and identical decoded pixels are recorded separately. Conflicting labels for identical pixels are excluded. Deduplication precedes the seeded stratified 70/15/15 split; any original folder splits are deliberately repartitioned. Near duplicates and source overlap can still remain.

For **each** figure below, write 2–3 sentences based on the actual result: describe the visible pattern, compare the classes, and explain a modeling implication. Class-dependent dimensions/compression can become shortcuts. Inspect the samples to confirm that folder labels match the images. See the report worksheet for prompts.

In [ ]:
import pandas as pd
from IPython.display import display, Image
for name in ['class_counts.csv', 'summary_statistics.csv']:
    print(name)
    display(pd.read_csv(DATA_EVIDENCE / name))
print((DATA_EVIDENCE / 'audit_summary.json').read_text())
print((DATA_EVIDENCE / 'split_summary.json').read_text())
for name in ['samples.png', 'width_histogram.png', 'height_histogram.png', 'size_bytes_histogram.png']:
    display(Image(filename=str(DATA_EVIDENCE / name)))

## 1.4 — DVC toy: run the actual round trip
This creates a new isolated Git repository, commits both CSV versions, pushes both to a local-folder DVC remote, restores the older version, deletes only the toy CSV and toy cache, then pulls and verifies the original content. Capture the command output and commit hashes. Run once; if rerunning, use a new destination name.

Local storage is explicitly allowed for the DVC remote. To satisfy the separate DagsHub commit evidence, push this toy repository's `main` branch to a **separate empty DagsHub toy repository** using the instructions in `docs/DVC_GUIDE.md`. The toy commits use a clearly labeled demo author; set your own identity for the actual project.

In [ ]:
TOY_DIR = PROJECT_DIR / 'dvc_toy'
run_module('scripts.dvc_demo', '--destination', TOY_DIR)

## 1.4 — Version the actual dataset
Follow `docs/DVC_GUIDE.md` before claiming this part complete. DVC-track the original Kaggle dataset, the processed images and split manifests; push bytes to persistent Drive storage or DagsHub storage. Commit the `.dvc` pointers and `.dvc/config` in your named project repository. The dataset download itself does not count as DVC versioning.

DVC pointers hold a content checksum and path, plus metadata such as byte size and file count. Git stores these small text pointers; DVC storage contains the large image bytes. Large binaries inflate Git history and are difficult to diff. Your audit SHA-256 is separate from DVC's object checksum.

In [ ]:
print((WORK_DIR / 'docs/DVC_GUIDE.md').read_text())

### Run actual dataset versioning with the selected storage
Enter your Git identity, then run once. This copies the raw dataset (extra disk space required), DVC-tracks raw/processed/splits, pushes data to the selected storage and backs up committed Git metadata there. With local fallback this backup is temporary: publish to DagsHub/storage before runtime deletion. The evidence ZIP excludes full DVC caches and Git repositories. It refuses to overwrite an existing Git repository; if interrupted, follow the guide to finish the existing repository. Publishing commits to your DagsHub account remains a separate authenticated step.

In [ ]:
GIT_NAME = "YOUR NAME"
GIT_EMAIL = "YOUR GIT EMAIL"
if not DRIVE_MOUNTED:
    print('Local DVC storage is temporary; use the guide to publish data and commits to DagsHub.')
assert GIT_NAME != "YOUR NAME" and GIT_EMAIL != "YOUR GIT EMAIL", "Enter your Git identity"
run_module('scripts.version_dataset', '--raw', DATA_ROOT, '--splits', DATA_EVIDENCE,
           '--persistent', PROJECT_DIR, '--git-name', GIT_NAME, '--git-email', GIT_EMAIL)

## Validate training PNGs before fitting
This checks TensorFlow decoding and audited pixel hashes for every split image. Any incompatible or incomplete PNG is losslessly regenerated from the audited original using TensorFlow's encoder, validated, then atomically replaced. No images are dropped and split lists stay unchanged. This is file integrity checking, not test prediction/evaluation. Repairs are recorded under `png_integrity`. If DVC already tracks the processed files, update its pointer and push the repaired bytes afterward.

In [ ]:
run_module('scripts.repair_processed', '--raw', DATA_ROOT, '--processed', PROCESSED,
           '--splits', DATA_EVIDENCE, '--report', PROJECT_DIR / 'png_integrity')

## 2.1–2.4 — Pipeline and models
`tf.data`: read saved split paths/labels → decode normalized RGB copies → resize with padding to 224×224 → shuffle training only → batch → prefetch. Padding preserves aspect ratio but introduces borders; augmentation can perturb those borders. Moderate flip/rotation/zoom/contrast augmentation runs inside each model **only when training**. Validation/test order and pixels are deterministic.

**Scratch CNN:** three Conv→BatchNorm→ReLU→Pool blocks learn local patterns while reducing spatial resolution. Global average pooling limits parameters; a 64-unit dense layer combines features and dropout regularizes the classifier. One sigmoid returns P(AI). Binary cross entropy matches the binary output; Adam is a practical adaptive optimizer. Accuracy/AUC/precision/recall are monitored. Training precision/recall use 0.5; final metrics use the validation-selected threshold.

**Transfer:** MobileNetV2 ImageNet weights, no classifier, global average pooling, dropout and sigmoid. Embedded rescaling maps [0,255] to [-1,1], exactly the MobileNetV2 preprocessing. Train a frozen backbone first, then unfreeze only weighted non-BatchNorm layers in `block_16` and its terminal `Conv_1` projection at 1e-5 (100× below the head LR). BatchNorm stays frozen/in inference mode, and the model is recompiled after changing trainability. The standalone frozen and fine-tuned stages each save curves and trainable-layer lists.

Fixed seeds and deterministic TensorFlow ops improve reproducibility; hardware/library changes can still alter results. Early stopping, validation-loss checkpointing and plateau LR reduction run in each main stage. Best epoch is the minimum validation-loss epoch, marked on the curves.

## 2.7 — Keras Tuner (included in the run below)
The Fashion-MNIST toy tunes units and learning rate. VisionGuard runs **8 random-search trials** over learning rate, dropout, and the number of unfrozen weighted layers restricted to the last block. Each starts from the same frozen-head checkpoint. Saved marginal mean-loss tables help discuss which setting appeared most influential; eight confounded trials cannot prove causal importance. Repeated selection against validation may overfit that split, so it must never use test feedback.

## Run training, tuning and final evaluation
Defaults: 20 scratch epochs, 10 frozen-head epochs, 15 fine-tune epochs, up to 5 epochs per tuner trial. Early stopping may finish sooner. Runtime depends on the downloaded dataset and GPU; budget for multiple training runs and keep the tab connected. With local storage, download results as soon as training finishes. If GPU memory is exhausted, lower `BATCH_SIZE` before starting.

The script resumes completed stages/checkpoints and tuner state after a pre-test interruption. Use the **same settings and data** when resuming. If trials failed, fix the cause and restart the tuner in a fresh pre-test directory; eight completed trials are required. Once `TEST_EVALUATION_STARTED.json` exists, it refuses to rerun: reuse saved test probabilities/figures and do not retune based on test results.

In [ ]:
TRAINING_OUT = PROJECT_DIR / 'training'
BATCH_SIZE = 32
run_module('scripts.run_training', '--root', PROCESSED, '--splits', DATA_EVIDENCE,
           '--out', TRAINING_OUT, '--batch', BATCH_SIZE, '--seed', SEED)

## 2.5 — Evaluation and decision costs
AI is the positive class (1), real is 0. For this product a false accusation of a real photograph can damage trust and reputation, so the predeclared example cost is 2× for real→AI versus AI→real. This is a design assumption, not an empirically established business cost. The threshold minimizes `2*FP + FN` using **validation only**, with conservative ties. Transfer candidates are chosen by validation loss before opening test.

Each final model makes one test inference pass. All metrics, confusion matrix, ROC curve and error images reuse those predictions. Accuracy, precision, recall, F1 and ROC-AUC are saved for both models. ROC-AUC uses continuous probabilities. A detector trained on this dataset can fail on new generators or image sources; do not present it as proof of authenticity.

In [ ]:
print('Project:', PROJECT_NAME, 'UTC:', datetime.now(timezone.utc).isoformat())
display(pd.read_csv(TRAINING_OUT / 'comparison.csv'))
print((TRAINING_OUT / 'selection_before_test.json').read_text())
for stage in ['baseline', 'head', 'finetune']:
    print(stage, (TRAINING_OUT / stage / 'best_epoch.json').read_text())
    display(Image(filename=str(TRAINING_OUT / stage / 'curves.png')))
display(Image(filename=str(TRAINING_OUT / 'augmented_batch.png')))
for model in ['baseline', 'transfer']:
    for figure in ['confusion_matrix.png', 'roc.png', 'misclassified.png']:
        display(Image(filename=str(TRAINING_OUT / 'evaluation' / model / figure)))

## 2.6 — Complete the image-by-image error analysis
Open each model's `evaluation/<model>/error_analysis.csv`. For each of the 10 displayed errors, enter a specific visible observation and a testable hypothesis. Examples to investigate include blur, painting-like texture, watermarks, unusual crops or severe resizing. These are prompts, not claims about the actual images. If fewer than 10 errors exist, report that count honestly and do not duplicate errors.

Save the completed CSVs on Drive and include the grid and hypotheses in your report. The notebook deliberately leaves these observations blank until you inspect the images.

In [ ]:
display(pd.read_csv(TRAINING_OUT / 'tuner_toy' / 'trials.csv'))
display(pd.read_csv(TRAINING_OUT / 'tuner' / 'trials.csv'))
print((TRAINING_OUT / 'tuner' / 'best_hyperparameters.json').read_text())
for param in ['learning_rate', 'dropout', 'unfrozen_layers']:
    print(param)
    display(pd.read_csv(TRAINING_OUT / 'tuner' / f'{param}_association.csv'))

## Download evidence and source
The archive includes CSVs, plots, model checkpoints, tuner state, exact package versions and source. It excludes the large dataset and DVC toy cache. Download size may be substantial due to model checkpoints. If Drive mounted successfully, it is also preserved in your Google Drive project folder. Otherwise it exists only on the runtime disk: download it before disconnecting/deleting the runtime.

Use `docs/REPORT_WORKSHEET.md` to assemble Tasks 1 and 2 in your own words, adding real screenshots with project name/date context, DagsHub evidence, interpretations and observed failures. Disclose AI assistance as required by the assignment. Tasks 3–8 are outside this deliverable.

In [ ]:
import zipfile
archive = Path('/content') / f'{PROJECT_NAME}_tasks_1_2_evidence.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in [PROJECT_DIR / 'environment', DATA_EVIDENCE, TRAINING_OUT, PROJECT_DIR / 'source', PROJECT_DIR / 'png_integrity']:
        for path in folder.rglob('*'):
            if path.is_file():
                z.write(path, path.relative_to(PROJECT_DIR))
    for name in ['result.json', 'transcript.txt']:
        path = TOY_DIR / name
        if path.exists(): z.write(path, 'dvc_toy/' + name)
print(archive)
from google.colab import files
files.download(str(archive))